> **軽量配布版**：ダウンロードしやすいよう、過去の出力画像と動画の埋め込みを除去しています。解説・コードは変更していません。上から実行すると結果とStage 1の動画を再生成できます。MP4の保存にはffmpegが必要です。


# 1次元Euler方程式で学ぶ流体スキームの比較
### LF・LW・Roe・LLF（Rusanov）・HLL・HLLC

このNotebookでは、**数値拡散、数値分散、接触不連続面、衝撃波、正値性**を分けて調べます。
まず基本6方式を比較し、次にRoe・LLF・HLL・HLLCの再構成と時間積分を統一して比較します。
**グラフを眺めるだけでなく、厳密解に対する誤差、保存則、失敗時刻も測定**します。

| 比較段階 | 空間再構成・時間積分 | 目的 |
|:--|:--|:--|
| Stage 1 | LF・区分一定のRoe/LLF/HLL/HLLCは1次。LWはRichtmyerの2段階法で滑らかな解に対して2次 | 古典的な方式と基本的な数値流束の性質 |
| Stage 2 | 4つのリーマンソルバーに共通のMUSCL再構成＋SSP-RK2 | 高次精度化後の数値流束の違い |

**使い方：** 最初の設定セルを確認し、上から順に実行してください。計算済みの図と表を保存した版です。
Notebook単独で動作し、外部データ、Clawpack、Fortranコンパイラ、GPU、インターネット接続は不要です。
必要なのはPython、NumPy、Matplotlib、Jupyterです。SciPy、pandas、Numbaは要求しません。

**失敗は教材の一部です。** 負の密度・圧力が発生した計算は例外を捕捉して停止し、次のスキームへ進みます。
密度・圧力のfloor、時間刻みの自動再試行、別ソルバーへの自動切替は使用しません。
失敗した途中時刻の解を、他方式の最終時刻の解として描くこともありません。

作成版：2026-11-07 / v1.0。数式・実装の参考文献は末尾にまとめています。

**更新：末尾に[Stage 1の動画比較](#stage-1-animation)を追加しました。動画だけを実行する独立版Notebookも同梱しています。**

## 0. 設定と実行環境

変更する主な変数はこのセルにまとめました。最初は$\gamma=1.4$、CFL数$0.4$を維持してください。
`n_basic`は基本比較、`n_convergence`は滑らかな波の収束試験、`n_reference`はShu–Osher問題の数値参照解の格子数です。

`run_shu_osher=False`にすると参照解を含むShu–Osher計算を省略できます。
`run_extras=False`にするとCFL依存性、エントロピー補正の有無、Lax衝撃波管の追加実験を省略できます。
`limiter="mc"`に変更するとminmodより制限の弱いMCリミッターを試せます。
`"unlimited"`は滑らかな問題の追加実験専用と考えてください。

In [ ]:
from dataclasses import dataclass, replace
from functools import lru_cache
from typing import Optional, Tuple
import time
import numpy as np

In [ ]:
CFG = {
    "gamma": 1.4,
    "cfl": 0.4,
    "n_basic": 200,
    "n_convergence": (100, 200, 400, 800),
    "n_shu_osher": 400,
    "n_reference": (1600, 3200),
    "limiter": "minmod",
    "reference_limiter": "mc",
    "wave_speeds": "einfeldt",
    "roe_entropy": 0.1,
    "run_convergence": True,
    "run_shu_osher": True,
    "run_extras": True,
    "export_results": True,
}

if "RUN_CACHE" in globals():
    RUN_CACHE.clear()

import sys
import matplotlib
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Default settings:", CFG)

## 1. 方程式、変数、保存形

無粘性・断熱の理想気体を扱います。量はすべて無次元です。$E$は**単位体積当たりの全エネルギー**、$e$は**単位質量当たりの内部エネルギー**です。

$$
\frac{\partial\boldsymbol U}{\partial t}+\frac{\partial\boldsymbol F}{\partial x}=0,
\qquad
\boldsymbol U=\begin{pmatrix}\rho\\\rho u\\E\end{pmatrix},
\qquad
\boldsymbol F=\begin{pmatrix}\rho u\\\rho u^2+p\\u(E+p)\end{pmatrix}.
$$
$$
E=\frac{p}{\gamma-1}+\frac12\rho u^2,\qquad
c=\sqrt{\frac{\gamma p}{\rho}},\qquad e=\frac{p}{(\gamma-1)\rho}.
$$

特性速度は$u-c,\ u,\ u+c$です。中央の波が接触・エントロピー波を担います。[R1]

コードでは`U.shape == (N, 3)`とし、`U[:, 0]`が密度、`U[:, 1]`が運動量密度、`U[:, 2]`が全エネルギー密度です。
基本変数`W`は$(\rho,u,p)$の順です。境界流束を差し引く更新式を全方式で共有します。

$$
\overline{\boldsymbol U}^{\,n+1}_i=\overline{\boldsymbol U}^{\,n}_i-
\frac{\Delta t}{\Delta x}(\boldsymbol F^*_{i+1/2}-\boldsymbol F^*_{i-1/2}).
$$

ここで格子上の未知量は**点値ではなくセル平均**です。
一様格子上で区分一定の状態を用いる場合、以下のLF/LWの保存形実装は対応する古典的な差分更新になります。

In [ ]:
class NumericalFailure(RuntimeError):
    """An inadmissible trial state; the integrator retains the last accepted state."""
    def __init__(self, stage, U=None, gamma=1.4, detail="", primitive=None):
        self.stage = stage
        self.min_rho = np.nan
        self.min_p = np.nan
        self.bad_index = None
        if U is not None:
            rho = np.asarray(U)[..., 0]
            with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                pressure = (gamma - 1.0) * (U[..., 2] - 0.5 * U[..., 1]**2 / rho)
            self.min_rho = float(np.min(rho))
            self.min_p = float(np.min(pressure))
            bad = (~np.all(np.isfinite(U), axis=-1)) | (~np.isfinite(pressure)) | (rho <= 0.0) | (pressure <= 0.0)
            indices = np.flatnonzero(bad)
            self.bad_index = int(indices[0]) if len(indices) else None
        if primitive is not None:
            W = np.asarray(primitive)
            self.min_rho = float(np.min(W[..., 0]))
            self.min_p = float(np.min(W[..., 2]))
            bad = (~np.all(np.isfinite(W), axis=-1)) | (W[..., 0] <= 0) | (W[..., 2] <= 0)
            indices = np.flatnonzero(bad)
            self.bad_index = int(indices[0]) if len(indices) else None
        message = f"{stage}: {detail} min(rho)={self.min_rho:.5e}, min(p)={self.min_p:.5e}"
        super().__init__(message.strip())


def prim_to_cons(W: np.ndarray, gamma: float = 1.4) -> np.ndarray:
    """基本変数 W=(rho,u,p) を保存変数 U=(rho,rho*u,E) に変換する。"""
    W = np.asarray(W, dtype=float)
    rho, u, p = W[..., 0], W[..., 1], W[..., 2]
    if not np.isfinite(gamma) or gamma <= 1.0 or np.any(rho <= 0.0) or np.any(p <= 0.0) or not np.all(np.isfinite(W)):
        raise ValueError("gamma > 1, finite rho/u/p, rho > 0 and p > 0 are required.")
    return np.stack((rho, rho * u, p / (gamma - 1.0) + 0.5 * rho * u**2), axis=-1)


def cons_to_prim(U: np.ndarray, gamma: float = 1.4, check: bool = True, stage: str = "state") -> np.ndarray:
    """保存変数から基本変数を求める。非物理的な値を切り上げる操作はしない。"""
    U = np.asarray(U, dtype=float)
    rho = U[..., 0]
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        u = U[..., 1] / rho
        p = (gamma - 1.0) * (U[..., 2] - 0.5 * U[..., 1] * u)
    W = np.stack((rho, u, p), axis=-1)
    if check and (not np.all(np.isfinite(W)) or np.any(rho <= 0.0) or np.any(p <= 0.0)):
        raise NumericalFailure(stage, U, gamma)
    return W


def physical_flux(U: np.ndarray, gamma: float = 1.4, stage: str = "flux state") -> np.ndarray:
    W = cons_to_prim(U, gamma, stage=stage)
    rho, u, p = W[..., 0], W[..., 1], W[..., 2]
    return np.stack((rho * u, rho * u**2 + p, u * (U[..., 2] + p)), axis=-1)


def max_signal_speed(U: np.ndarray, gamma: float = 1.4) -> float:
    W = cons_to_prim(U, gamma)
    return float(np.max(np.abs(W[..., 1]) + np.sqrt(gamma * W[..., 2] / W[..., 0])))

## 2. 数値流束の定義

### 2.1 古典的LF、2段階LW、LLFを区別する

左・右の状態を$\boldsymbol U_L,\boldsymbol U_R$、流束を$\boldsymbol F_L,\boldsymbol F_R$とします。
古典的LFでは

$$
\boldsymbol F^*_{\rm LF}=\frac12(\boldsymbol F_L+\boldsymbol F_R)
-\frac12\frac{\Delta x}{\Delta t}(\boldsymbol U_R-\boldsymbol U_L).
$$

一方、LLF（Rusanov）では

$$
\boldsymbol F^*_{\rm LLF}=\frac12(\boldsymbol F_L+\boldsymbol F_R)
-\frac12\alpha_{LR}(\boldsymbol U_R-\boldsymbol U_L),\qquad
\alpha_{LR}=\max(|u_L|+c_L,|u_R|+c_R).
$$

**LFの散逸係数は$\Delta x/\Delta t$、LLFは局所的な波速です。** ここを同一実装にしないことが比較の要点です。
LLFを時間積分した解にも時間刻み依存性はあります。「流束の係数に$\Delta t$が入らない」と「解が$\Delta t$に依存しない」は別です。

本NotebookのLWは、非線形Euler方程式に対する**Richtmyerの2段階Lax–Wendroff法**です。

$$
\boldsymbol U_{LR}^{n+1/2}=\frac12(\boldsymbol U_L+\boldsymbol U_R)
-\frac{\Delta t}{2\Delta x}(\boldsymbol F_R-\boldsymbol F_L),\qquad
\boldsymbol F^*_{\rm LW}=\boldsymbol F(\boldsymbol U_{LR}^{n+1/2}).
$$

非線形方程式には複数のLW型定式化があります。この実装を指して`LW`と表記します。
人工粘性・リミッターを付けないため、不連続面近傍の分散的な振動を観察できます。
半時間ステップの状態が非物理的になった場合も、その時点で停止します。

### 2.2 Roe流束とエントロピー補正

Roe平均を

$$
\widetilde u=\frac{\sqrt{\rho_L}u_L+\sqrt{\rho_R}u_R}{\sqrt{\rho_L}+\sqrt{\rho_R}},\qquad
\widetilde H=\frac{\sqrt{\rho_L}H_L+\sqrt{\rho_R}H_R}{\sqrt{\rho_L}+\sqrt{\rho_R}},\qquad
\widetilde c^2=(\gamma-1)\left(\widetilde H-\frac12\widetilde u^2\right)
$$

と定義します。$H=(E+p)/\rho$です。$\Delta\boldsymbol U=\sum_k\beta_k\boldsymbol r_k$と分解すると、

$$
\boldsymbol F^*_{\rm Roe}=\frac12(\boldsymbol F_L+\boldsymbol F_R)
-\frac12\sum_{k=1}^3|\widetilde\lambda_k|\,\beta_k\boldsymbol r_k
$$

となります。具体的な波の強度と固有ベクトルは次のコードに明示しています。[R2]

本Notebookでは、音響波$\widetilde u\pm\widetilde c$の絶対値に、次のHarten型の簡易正則化を選びます。

$$
|\lambda|_\delta=
\begin{cases}
|\lambda|,&|\lambda|\ge\delta,\\
(\lambda^2+\delta^2)/(2\delta),&|\lambda|<\delta,
\end{cases}
\qquad \delta=\varepsilon_{\rm ent}\widetilde c.
$$

標準値は$\varepsilon_{\rm ent}=0.1$です。`roe_entropy=0.0`なら補正しません。
**接触波$\widetilde u$にはこの補正を加えません。**
この簡易補正は低密度・低圧力の正値性対策ではなく、あらゆる希薄波での保証を与えるものでもありません。[R7]

### 2.3 HLLとHLLC：外側波速を共通にして比較する

HLLは最も左向きと右向きの波速$S_L,S_R$で挟まれた領域を一つの状態で近似します。

$$
\boldsymbol F^*_{\rm HLL}=
\begin{cases}
\boldsymbol F_L,&0\le S_L,\\
\dfrac{S_R\boldsymbol F_L-S_L\boldsymbol F_R+S_LS_R(\boldsymbol U_R-\boldsymbol U_L)}{S_R-S_L},&S_L<0<S_R,\\
\boldsymbol F_R,&S_R\le0.
\end{cases}
$$

標準設定はEinfeldtの波速推定

$$
S_L=\min(u_L-c_L,\widetilde u-\widetilde c),\qquad
S_R=\max(u_R+c_R,\widetilde u+\widetilde c)
$$

です。したがって図では**HLL (HLLE)**と表示します。[R2]
`wave_speeds="davis"`では両側の$u\pm c$の最小・最大を用います。
`"symmetric"`では$S_L=-\alpha_{LR},S_R=\alpha_{LR}$として、HLLがLLFと一致することを確認できます。

HLLCはHLLの中央領域を、接触波$S_*$で二つに分けます。

$$
S_* = \frac{p_R-p_L+\rho_Lu_L(S_L-u_L)-\rho_Ru_R(S_R-u_R)}
{\rho_L(S_L-u_L)-\rho_R(S_R-u_R)}.
$$

$K=L,R$について

$$
\rho_{*K}=\rho_K\frac{S_K-u_K}{S_K-S_*},\qquad
E_{*K}=\rho_{*K}\left[\frac{E_K}{\rho_K}+(S_*-u_K)
\left(S_*+\frac{p_K}{\rho_K(S_K-u_K)}\right)\right]
$$

を使い、$\boldsymbol U_{*K}=(\rho_{*K},\rho_{*K}S_*,E_{*K})$を構成します。
中央領域の流束は$\boldsymbol F_K+S_K(\boldsymbol U_{*K}-\boldsymbol U_K)$です。[R3, R6]

ここではHLLとHLLCに同じ$S_L,S_R$を使います。
この選択で完走したという実験結果を、高次再構成や任意の波速推定を含めた一般的な正値性保証と混同しないでください。

In [ ]:
RIEMANN_SOLVERS = ("Roe", "LLF", "HLL", "HLLC")
ALL_SOLVERS = ("LF", "LW", "Roe", "LLF", "HLL", "HLLC")


def roe_averages(UL: np.ndarray, UR: np.ndarray, WL: np.ndarray, WR: np.ndarray, gamma: float):
    rootL, rootR = np.sqrt(WL[..., 0]), np.sqrt(WR[..., 0])
    u = (rootL * WL[..., 1] + rootR * WR[..., 1]) / (rootL + rootR)
    HL, HR = (UL[..., 2] + WL[..., 2]) / WL[..., 0], (UR[..., 2] + WR[..., 2]) / WR[..., 0]
    H = (rootL * HL + rootR * HR) / (rootL + rootR)
    a2 = (gamma - 1.0) * (H - 0.5 * u**2)
    if np.any(a2 <= 0.0) or not np.all(np.isfinite(a2)):
        raise NumericalFailure("Roe average", detail="Nonpositive or nonfinite sound-speed squared.")
    return rootL * rootR, u, H, np.sqrt(a2)


def signal_bounds(UL, UR, WL, WR, gamma: float, method: str = "einfeldt"):
    """HLL/HLLCに同じ外側波速を使う。einfeldtの場合のHLLをHLLEと表示する。"""
    uL, uR = WL[..., 1], WR[..., 1]
    aL, aR = np.sqrt(gamma * WL[..., 2] / WL[..., 0]), np.sqrt(gamma * WR[..., 2] / WR[..., 0])
    if method == "einfeldt":
        _, u, _, a = roe_averages(UL, UR, WL, WR, gamma)
        SL, SR = np.minimum(uL - aL, u - a), np.maximum(uR + aR, u + a)
    elif method == "davis":
        SL, SR = np.minimum(uL - aL, uR - aR), np.maximum(uL + aL, uR + aR)
    elif method == "symmetric":
        amax = np.maximum(np.abs(uL) + aL, np.abs(uR) + aR)
        SL, SR = -amax, amax
    else:
        raise ValueError("Wave speeds must be 'einfeldt', 'davis', or 'symmetric'.")
    return SL, SR


def entropy_abs(lam: np.ndarray, delta: np.ndarray) -> np.ndarray:
    """音響波だけに用いるHarten型の二次正則化。delta=0なら補正なし。"""
    value = np.abs(lam).copy()
    mask = (delta > 0.0) & (value < delta)
    value[mask] = 0.5 * (lam[mask]**2 / delta[mask] + delta[mask])
    return value


def numerical_flux(UL, UR, solver: str, gamma: float = 1.4, dt: Optional[float] = None,
                   dx: Optional[float] = None, wave_speeds: str = "einfeldt", roe_entropy: float = 0.1):
    """各セル境界の数値流束を返す。LLFのalphaは局所的、LFのalphaはdx/dt。"""
    if solver not in ALL_SOLVERS:
        raise ValueError(f"Unknown solver: {solver}")
    UL, UR = np.asarray(UL, dtype=float), np.asarray(UR, dtype=float)
    if UL.ndim != 2 or UL.shape != UR.shape or UL.shape[1] != 3:
        raise ValueError("Interface states must both have shape (number_of_faces, 3).")
    WL, WR = cons_to_prim(UL, gamma, stage="left interface"), cons_to_prim(UR, gamma, stage="right interface")
    FL, FR = physical_flux(UL, gamma), physical_flux(UR, gamma)
    rhoL, uL, pL = WL[..., 0], WL[..., 1], WL[..., 2]
    rhoR, uR, pR = WR[..., 0], WR[..., 1], WR[..., 2]
    aL, aR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    jump = UR - UL
    if solver in ("LF", "LW"):
        if dt is None or dx is None or dt <= 0.0 or dx <= 0.0:
            raise ValueError("LF and LW require positive dt and dx.")
        if solver == "LF":
            return 0.5 * (FL + FR) - 0.5 * (dx / dt) * jump
        Uhalf = 0.5 * (UL + UR) - 0.5 * (dt / dx) * (FR - FL)
        return physical_flux(Uhalf, gamma, stage="LW half-step predictor")
    if solver == "LLF":
        alpha = np.maximum(np.abs(uL) + aL, np.abs(uR) + aR)
        return 0.5 * (FL + FR) - 0.5 * alpha[..., None] * jump
    if solver == "Roe":
        rho, u, H, a = roe_averages(UL, UR, WL, WR, gamma)
        drho, du, dp = rhoR - rhoL, uR - uL, pR - pL
        beta1, beta3 = (dp - rho * a * du) / (2.0 * a**2), (dp + rho * a * du) / (2.0 * a**2)
        beta2 = drho - dp / a**2
        r1 = np.stack((np.ones_like(u), u - a, H - u * a), axis=-1)
        r2 = np.stack((np.ones_like(u), u, 0.5 * u**2), axis=-1)
        r3 = np.stack((np.ones_like(u), u + a, H + u * a), axis=-1)
        delta = roe_entropy * a
        d1, d2, d3 = entropy_abs(u - a, delta), np.abs(u), entropy_abs(u + a, delta)
        dissipation = (beta1 * d1)[..., None] * r1 + (beta2 * d2)[..., None] * r2 + (beta3 * d3)[..., None] * r3
        return 0.5 * (FL + FR - dissipation)
    SL, SR = signal_bounds(UL, UR, WL, WR, gamma, wave_speeds)
    if solver == "HLL":
        Fmiddle = (SR[..., None] * FL - SL[..., None] * FR + (SL * SR)[..., None] * jump) / (SR - SL)[..., None]
        return np.where((SL >= 0.0)[..., None], FL, np.where((SR <= 0.0)[..., None], FR, Fmiddle))
    # HLLC: 音響波の間に接触波 Sstar を復元する。不要な超音速側のstar状態は計算しない。
    result = np.where((SL >= 0.0)[..., None], FL, FR).copy()
    middle = (SL < 0.0) & (SR > 0.0)
    if not np.any(middle):
        return result
    sl, sr = SL[middle], SR[middle]
    rl, rr, vl, vr, pl, pr = rhoL[middle], rhoR[middle], uL[middle], uR[middle], pL[middle], pR[middle]
    numerator = pr - pl + rl * vl * (sl - vl) - rr * vr * (sr - vr)
    denominator = rl * (sl - vl) - rr * (sr - vr)
    if np.any(denominator == 0.0):
        raise NumericalFailure("HLLC contact speed", detail="Zero denominator.")
    sm = numerator / denominator
    if np.any(sl >= sm) or np.any(sm >= sr):
        raise NumericalFailure("HLLC wave ordering", detail="Expected SL < Sstar < SR.")
    rsl, rsr = rl * (sl - vl) / (sl - sm), rr * (sr - vr) / (sr - sm)
    esl = rsl * (UL[middle, 2] / rl + (sm - vl) * (sm + pl / (rl * (sl - vl))))
    esr = rsr * (UR[middle, 2] / rr + (sm - vr) * (sm + pr / (rr * (sr - vr))))
    Usl, Usr = np.column_stack((rsl, rsl * sm, esl)), np.column_stack((rsr, rsr * sm, esr))
    Fsl, Fsr = FL[middle] + sl[:, None] * (Usl - UL[middle]), FR[middle] + sr[:, None] * (Usr - UR[middle])
    result[middle] = np.where((sm >= 0.0)[:, None], Fsl, Fsr)
    return result

## 3. テスト問題と初期条件

すべて$\gamma=1.4$の設定です。$W=(\rho,u,p)$と表記します。
矩形波のセル内占有率と正弦波のセル平均を解析的に求め、格子点で初期条件を単純にサンプルする誤差を避けます。

| キー | 領域・時刻 | 初期条件の要点 | 境界 |
|:--|:--|:--|:--|
| `entropy` | $[0,1],\ t=1$ | $\rho=1+0.2\sin(2\pi x),\ u=p=1$ | 周期 |
| `contact` | $[0,1],\ t=0.25$ | $0.25\le x<0.45$で$\rho=2$、他は1。$u=p=1$ | 周期 |
| `contact_static` | $[0,1],\ t=0.25$ | 同じ矩形密度、$u=0,\ p=1$ | 周期 |
| `sod` | $[0,1],\ x_0=0.5,\ t=0.2$ | $W_L=(1,0,1),\ W_R=(0.125,0,0.1)$ | 流出 |
| `double_rarefaction` | $[-0.5,0.5],\ x_0=0,\ t=0.15$ | $W_L=(1,-2,0.4),\ W_R=(1,2,0.4)$ | 流出 |
| `shu_osher` | $[-5,5],\ x_0=-4,\ t=1.8$ | 左は衝撃波通過後、右は$\rho=1+0.2\sin(5x),\ u=0,\ p=1$ | 左固定流入・右流出 |
| `transonic` | $[0,1],\ x_0=0.3,\ t=0.2$ | $W_L=(1,0.75,1),\ W_R=(0.125,0,0.1)$ | 流出 |
| `lax` | $[-0.5,0.5],\ x_0=0,\ t=0.14$ | $W_L=(0.445,0.698,3.528),\ W_R=(0.5,0,0.571)$ | 流出 |

SodとShu–Osherの設定は文献・公式テスト例と対応します。[R4, R5]
Shu–Osherの左状態は$\rho_L=27/7,\ u_L=20\sqrt{1.4}/9,\ p_L=31/3$とし、よく用いられる小数値の丸めを避けています。
$\gamma$を変更した場合、この左状態が同じMach数の衝撃波に対応するとは限りません。

流出境界は端のセルをゴーストセルにコピーする簡単な条件で、完全な無反射境界ではありません。
指定した終了時刻では、Sodなどのリーマン問題の主要波面は境界に達しません。

In [ ]:
@dataclass(frozen=True)
class Problem:
    key: str
    title: str
    bounds: Tuple[float, float]
    final_time: float
    boundary: str
    left: Optional[Tuple[float, float, float]] = None
    right: Optional[Tuple[float, float, float]] = None
    x0: float = 0.5
    velocity: float = 1.0


PROBLEMS = {
    "entropy": Problem("entropy", "Smooth entropy-wave advection", (0.0, 1.0), 1.0, "periodic"),
    "contact": Problem("contact", "Moving contact discontinuities", (0.0, 1.0), 0.25, "periodic"),
    "contact_static": Problem("contact_static", "Stationary contact discontinuities", (0.0, 1.0), 0.25, "periodic", velocity=0.0),
    "sod": Problem("sod", "Sod shock tube", (0.0, 1.0), 0.2, "outflow", (1.0, 0.0, 1.0), (0.125, 0.0, 0.1)),
    "double_rarefaction": Problem("double_rarefaction", "Two strong rarefactions", (-0.5, 0.5), 0.15, "outflow", (1.0, -2.0, 0.4), (1.0, 2.0, 0.4), x0=0.0),
    "lax": Problem("lax", "Lax shock tube", (-0.5, 0.5), 0.14, "outflow", (0.445, 0.698, 3.528), (0.5, 0.0, 0.571), x0=0.0),
    "transonic": Problem("transonic", "Transonic rarefaction", (0.0, 1.0), 0.2, "outflow", (1.0, 0.75, 1.0), (0.125, 0.0, 0.1), x0=0.3),
    "shu_osher": Problem("shu_osher", "Shu-Osher shock / entropy-wave interaction", (-5.0, 5.0), 1.8, "inflow_outflow", (27.0 / 7.0, 20.0 * np.sqrt(1.4) / 9.0, 31.0 / 3.0), x0=-4.0),
}


def grid_for(problem: Problem, n: int):
    if not isinstance(n, (int, np.integer)) or n < 8:
        raise ValueError("n must be an integer >= 8.")
    if not np.isfinite(problem.bounds).all() or problem.bounds[1] <= problem.bounds[0]:
        raise ValueError("Problem bounds must be finite and increasing.")
    edges = np.linspace(problem.bounds[0], problem.bounds[1], n + 1)
    return edges, 0.5 * (edges[:-1] + edges[1:]), float(edges[1] - edges[0])


def advected_cell_average(problem: Problem, edges: np.ndarray, t: float, gamma: float):
    """正弦波/矩形波の厳密なセル平均。密度だけでなく保存変数を平均する。"""
    dx, x = np.diff(edges), 0.5 * (edges[:-1] + edges[1:])
    origin = problem.bounds[0]
    v, length = problem.velocity, problem.bounds[1] - problem.bounds[0]
    if problem.key == "entropy":
        rho = 1.0 + 0.2 * np.sinc(dx / length) * np.sin(2.0 * np.pi * (x - origin - v * t) / length)
    else:
        shift = (v * t) % length
        a, b = origin + 0.25 * length + shift, origin + 0.45 * length + shift
        overlap = np.zeros_like(x)
        for k in (-1, 0, 1):
            overlap += np.maximum(0.0, np.minimum(edges[1:], b + k * length) - np.maximum(edges[:-1], a + k * length))
        rho = 1.0 + overlap / dx
    W = np.column_stack((rho, np.full_like(rho, v), np.ones_like(rho)))
    return prim_to_cons(W, gamma)


def initial_cell_average(problem: Problem, edges: np.ndarray, gamma: float = 1.4) -> np.ndarray:
    if problem.key in ("entropy", "contact", "contact_static"):
        return advected_cell_average(problem, edges, 0.0, gamma)
    dx = np.diff(edges)
    frac = np.clip((problem.x0 - edges[:-1]) / dx, 0.0, 1.0)
    UL = prim_to_cons(np.array(problem.left), gamma)
    if problem.key != "shu_osher":
        UR = prim_to_cons(np.array(problem.right), gamma)
        return frac[:, None] * UL + (1.0 - frac)[:, None] * UR
    # 正弦波部分の密度を解析積分する。波と初期衝撃波が同じセルにあってもよい。
    lo, hi = np.maximum(edges[:-1], problem.x0), edges[1:]
    active = hi > lo
    integral_rho = np.zeros_like(dx)
    integral_rho[active] = (hi[active] - lo[active]) + 0.04 * (np.cos(5.0 * lo[active]) - np.cos(5.0 * hi[active]))
    U = frac[:, None] * UL
    U[:, 0] += integral_rho / dx
    U[:, 2] += (1.0 - frac) / (gamma - 1.0)
    return U

## 4. 厳密解と、比較に使うセル平均

エントロピー波と接触不連続面は、そのまま速度$u$で平行移動します。
Sod・二重希薄波・追加のリーマン問題は、星領域の圧力$p_*$を

$$
f_L(p_*)+f_R(p_*)+u_R-u_L=0
$$

から二分法で求めます。希薄波と衝撃波で分岐する$f_K$、自己相似変数$\xi=(x-x_0)/t$における解をコードに明示しています。[R1]
ここでの「厳密解」は厳密なリーマン解の構造に基づく数値評価で、非線形方程式の解法と数値積分の許容誤差は残ります。

**誤差は保存変数をセル平均した厳密解と比較します。** 波面の位置で積分区間を分割してからGauss積分するので、不連続を横切る単純な台形積分は使いません。
速度と圧力の比較基準は$W(\overline U_{\rm exact})$です。これは一般に$\overline{W(U_{\rm exact})}$とは異なります。

図には見やすさのため点値の厳密解を実線で示しますが、誤差表にはセル平均基準を使います。
Shu–Osherは厳密解を用いず、後で別に高解像度の**数値参照解**を計算します。

この実装は**物理的な真空を含むリーマン解には対応していません**。
真空条件$u_R-u_L\ge2(c_L+c_R)/(\gamma-1)$を検出すると明示的に停止します。
ここで使う二重希薄波問題は真空条件を満たさず、中央の密度・圧力は正です。

In [ ]:
@lru_cache(maxsize=128)
def riemann_structure(left: tuple, right: tuple, gamma: float = 1.4):
    """理想気体の厳密リーマン解。真空を含まない問題を対象にpressureを二分法で解く。"""
    rL, uL, pL = left
    rR, uR, pR = right
    if min(rL, rR, pL, pR) <= 0.0 or gamma <= 1.0:
        raise ValueError("Exact solver requires positive input states and gamma > 1.")
    cL, cR = np.sqrt(gamma * pL / rL), np.sqrt(gamma * pR / rR)
    if uR - uL >= 2.0 * (cL + cR) / (gamma - 1.0):
        raise ValueError("This exact solver does not implement a physical vacuum.")

    def wave_function(p, rho, pk, ck):
        if p > pk:
            return (p - pk) * np.sqrt((2.0 / ((gamma + 1.0) * rho)) / (p + (gamma - 1.0) * pk / (gamma + 1.0)))
        return (2.0 * ck / (gamma - 1.0)) * ((p / pk)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def residual(p):
        return wave_function(p, rL, pL, cL) + wave_function(p, rR, pR, cR) + uR - uL

    lower, upper = 0.0, max(pL, pR)
    for _ in range(100):
        if residual(upper) >= 0.0:
            break
        upper *= 2.0
    else:
        raise RuntimeError("Cannot bracket the star pressure.")
    for _ in range(180):
        p = 0.5 * (lower + upper)
        if residual(p) > 0.0:
            upper = p
        else:
            lower = p
        if upper - lower <= 2.0e-13 * max(upper, 1.0e-300):
            break
    ps = 0.5 * (lower + upper)
    us = 0.5 * (uL + uR + wave_function(ps, rR, pR, cR) - wave_function(ps, rL, pL, cL))
    b = (gamma - 1.0) / (gamma + 1.0)
    rLs = rL * ((ps / pL + b) / (b * ps / pL + 1.0) if ps > pL else (ps / pL)**(1.0 / gamma))
    rRs = rR * ((ps / pR + b) / (b * ps / pR + 1.0) if ps > pR else (ps / pR)**(1.0 / gamma))
    if ps > pL:
        sl = uL - cL * np.sqrt((gamma + 1.0) * ps / (2.0 * gamma * pL) + (gamma - 1.0) / (2.0 * gamma))
        left_speeds = (sl, sl)
    else:
        left_speeds = (uL - cL, us - cL * (ps / pL)**((gamma - 1.0) / (2.0 * gamma)))
    if ps > pR:
        sr = uR + cR * np.sqrt((gamma + 1.0) * ps / (2.0 * gamma * pR) + (gamma - 1.0) / (2.0 * gamma))
        right_speeds = (sr, sr)
    else:
        right_speeds = (us + cR * (ps / pR)**((gamma - 1.0) / (2.0 * gamma)), uR + cR)
    return {"pstar": ps, "ustar": us, "rhoLstar": rLs, "rhoRstar": rRs, "left_speeds": left_speeds, "right_speeds": right_speeds, "residual": residual(ps)}


def riemann_primitive(x: np.ndarray, t: float, problem: Problem, gamma: float = 1.4):
    original_shape = np.asarray(x).shape
    x = np.asarray(x, dtype=float).ravel()
    left, right = np.array(problem.left), np.array(problem.right)
    if t <= 0.0:
        return np.where((x < problem.x0)[:, None], left, right).reshape(original_shape + (3,))
    data = riemann_structure(tuple(problem.left), tuple(problem.right), gamma)
    ps, us = data["pstar"], data["ustar"]
    rL, uL, pL = left
    rR, uR, pR = right
    cL, cR = np.sqrt(gamma * pL / rL), np.sqrt(gamma * pR / rR)
    xi = (x - problem.x0) / t
    is_left = xi <= us
    W = np.where(is_left[:, None], np.array((data["rhoLstar"], us, ps)), np.array((data["rhoRstar"], us, ps)))
    lh, lt = data["left_speeds"]
    rt, rh = data["right_speeds"]
    W[xi < lh] = left
    W[xi > rh] = right
    if ps <= pL:
        fan = is_left & (xi >= lh) & (xi <= lt)
        c = (2.0 / (gamma + 1.0)) * (cL + 0.5 * (gamma - 1.0) * (uL - xi[fan]))
        W[fan, 0] = rL * (c / cL)**(2.0 / (gamma - 1.0))
        W[fan, 1] = (2.0 / (gamma + 1.0)) * (cL + 0.5 * (gamma - 1.0) * uL + xi[fan])
        W[fan, 2] = pL * (c / cL)**(2.0 * gamma / (gamma - 1.0))
    if ps <= pR:
        fan = (~is_left) & (xi >= rt) & (xi <= rh)
        c = (2.0 / (gamma + 1.0)) * (cR - 0.5 * (gamma - 1.0) * (uR - xi[fan]))
        W[fan, 0] = rR * (c / cR)**(2.0 / (gamma - 1.0))
        W[fan, 1] = (2.0 / (gamma + 1.0)) * (-cR + 0.5 * (gamma - 1.0) * uR + xi[fan])
        W[fan, 2] = pR * (c / cR)**(2.0 * gamma / (gamma - 1.0))
    return W.reshape(original_shape + (3,))


def exact_pointwise(problem: Problem, x: np.ndarray, t: float, gamma: float = 1.4):
    x = np.asarray(x, dtype=float)
    if problem.key == "shu_osher":
        raise ValueError("No exact solution is supplied for Shu-Osher.")
    if problem.key in ("entropy", "contact", "contact_static"):
        origin, end = problem.bounds
        coordinate = ((x - origin - problem.velocity * t) / (end - origin)) % 1.0
        rho = 1.0 + 0.2 * np.sin(2.0 * np.pi * coordinate) if problem.key == "entropy" else 1.0 + ((coordinate >= 0.25) & (coordinate < 0.45))
        return np.stack((rho, np.full_like(x, problem.velocity), np.ones_like(x)), axis=-1)
    return riemann_primitive(x, t, problem, gamma)


def exact_cell_average(problem: Problem, edges: np.ndarray, t: float, gamma: float = 1.4, nq: int = 12):
    """リーマン解は波面で区間を分割したGauss積分。保存変数のセル平均を返す。"""
    if problem.key in ("entropy", "contact", "contact_static"):
        return advected_cell_average(problem, edges, t, gamma)
    if problem.key == "shu_osher":
        raise ValueError("Use a numerical reference for Shu-Osher.")
    if t <= 0.0:
        return initial_cell_average(problem, edges, gamma)
    data = riemann_structure(tuple(problem.left), tuple(problem.right), gamma)
    speeds = np.array(data["left_speeds"] + (data["ustar"],) + data["right_speeds"])
    fronts = problem.x0 + t * speeds
    partitions = np.unique(np.r_[edges, fronts[(fronts > edges[0]) & (fronts < edges[-1])]])
    lo, hi = partitions[:-1], partitions[1:]
    nodes, weights = np.polynomial.legendre.leggauss(nq)
    points = 0.5 * (lo + hi)[:, None] + 0.5 * (hi - lo)[:, None] * nodes
    states = prim_to_cons(riemann_primitive(points, t, problem, gamma), gamma)
    integrals = 0.5 * (hi - lo)[:, None] * np.einsum("q,sqk->sk", weights, states)
    owners = np.searchsorted(edges, 0.5 * (lo + hi), side="right") - 1
    averaged = np.zeros((len(edges) - 1, 3))
    np.add.at(averaged, owners, integrals)
    return averaged / np.diff(edges)[:, None]

## 5. MUSCL再構成とSSP-RK2

Stage 2では基本変数$W=(\rho,u,p)$を再構成します。minmodでは

$$
\Delta W_i=\operatorname{minmod}(W_i-W_{i-1},W_{i+1}-W_i),\qquad
W_{i+1/2,L}=W_i+\frac12\Delta W_i,\quad W_{i+1/2,R}=W_{i+1}-\frac12\Delta W_{i+1}.
$$

`minmod`は二つの差分が同符号なら絶対値の小さい方、異符号なら0です。
`mc`は$\operatorname{minmod}((d_-+d_+)/2,2d_-,2d_+)$を用います。
コード中の$\Delta W_i$は**勾配そのものではなく、セル幅を掛けた増分**です。

空間離散化を$L(U)$とすると、SSP-RK2は

$$
U^{(1)}=U^n+\Delta t L(U^n),\qquad
U^{n+1}=\frac12U^n+\frac12\left[U^{(1)}+\Delta t L(U^{(1)})\right].
$$

各段階で境界条件を更新します。標準CFL数は0.4です。
MUSCLは滑らかな領域で2次精度を目指す一方、極値・不連続近傍ではリミッターが局所精度を下げます。
したがって、観測した収束次数が常に厳密に2になるとは限りません。
また、成分別のリミッターをEuler系に使っただけで、解全体のTVD性や正値性を無条件に保証したことにはなりません。
SSP型時間積分については[R8]を参照してください。

In [ ]:
def minmod(a: np.ndarray, b: np.ndarray):
    same_sign = np.sign(a) == np.sign(b)
    return np.where(same_sign, np.sign(a) * np.minimum(np.abs(a), np.abs(b)), 0.0)


def ghost_cells(U: np.ndarray, problem: Problem, gamma: float):
    """左右2個ずつのゴーストセルを作る。境界条件は各RK段階で更新する。"""
    if problem.boundary == "periodic":
        return np.concatenate((U[-2:], U, U[:2]), axis=0)
    if problem.boundary == "outflow":
        return np.concatenate((np.repeat(U[:1], 2, axis=0), U, np.repeat(U[-1:], 2, axis=0)), axis=0)
    if problem.boundary == "inflow_outflow":
        UL = prim_to_cons(np.array(problem.left), gamma)
        return np.concatenate((np.repeat(UL[None, :], 2, axis=0), U, np.repeat(U[-1:], 2, axis=0)), axis=0)
    raise ValueError(f"Unknown boundary condition: {problem.boundary}")


def interface_states(U: np.ndarray, problem: Problem, gamma: float, order: int, limiter: str):
    Ug = ghost_cells(U, problem, gamma)
    if order == 1:
        return Ug[1:-2].copy(), Ug[2:-1].copy()
    Wg = cons_to_prim(Ug, gamma, stage="MUSCL cell state")
    dm, dp = Wg[1:-1] - Wg[:-2], Wg[2:] - Wg[1:-1]
    slopes = np.zeros_like(Wg)
    if limiter == "minmod":
        slopes[1:-1] = minmod(dm, dp)
    elif limiter == "mc":
        slopes[1:-1] = minmod(0.5 * (dm + dp), minmod(2.0 * dm, 2.0 * dp))
    elif limiter == "unlimited":
        slopes[1:-1] = 0.5 * (dm + dp)
    else:
        raise ValueError("limiter must be 'minmod', 'mc', or 'unlimited'.")
    WL, WR = Wg[1:-2] + 0.5 * slopes[1:-2], Wg[2:-1] - 0.5 * slopes[2:-1]
    for state in (WL, WR):
        if not np.all(np.isfinite(state)) or np.any(state[:, 0] <= 0.0) or np.any(state[:, 2] <= 0.0):
            raise NumericalFailure("MUSCL reconstruction", detail="A reconstructed face rho or p is not positive.", primitive=state)
    return prim_to_cons(WL, gamma), prim_to_cons(WR, gamma)


def advance(U, problem: Problem, solver: str, dt: float, dx: float, gamma: float,
            order: int, limiter: str, wave_speeds: str, roe_entropy: float):
    """更新後の保存変数と、同じ数値流束で積算した領域への正味流入量を返す。"""
    UL, UR = interface_states(U, problem, gamma, order, limiter)
    F1 = numerical_flux(UL, UR, solver, gamma, dt, dx, wave_speeds, roe_entropy)
    U1 = U - (dt / dx) * (F1[1:] - F1[:-1])
    cons_to_prim(U1, gamma, stage="RK stage 1" if order == 2 else "full-step update")
    if order == 1:
        return U1, dt * (F1[0] - F1[-1])
    UL, UR = interface_states(U1, problem, gamma, order, limiter)
    F2 = numerical_flux(UL, UR, solver, gamma, dt, dx, wave_speeds, roe_entropy)
    Unew = 0.5 * U + 0.5 * (U1 - (dt / dx) * (F2[1:] - F2[:-1]))
    cons_to_prim(Unew, gamma, stage="RK stage 2")
    exchange = 0.5 * dt * ((F1[0] - F1[-1]) + (F2[0] - F2[-1]))
    return Unew, exchange

## 6. 時間積分、失敗判定、保存量の収支

時間刻みは各計算で

$$
\Delta t=C_{\rm CFL}\frac{\Delta x}{\max_i(|u_i|+c_i)}
$$

から求めます。**共通なのはCFL数と選択規則であり、時々刻々の$\Delta t$が全方式で同じとは限りません。**
終端で極端に短い最後のステップだけを取ると、LFでは$\Delta x/\Delta t$が急増します。
これを避けるため、残り時間がCFL上限の2ステップ以内になったら残りを1または2等分します。この規則は全方式に共通です。

非周期境界では領域内の総量が変化しても保存則違反とは限りません。$Q=\sum_iU_i\Delta x$として

$$
R_Q=Q(t)-Q(0)-\int_0^t(F_{\rm left}^*-F_{\rm right}^*)\,dt'
$$

を測定します。SSP-RK2の境界流束も段階の重み$1/2,1/2$で積算します。
表の`balance_max`は質量・運動量・全エネルギーの$|R_Q|$の最大値で、無次元の絶対残差です。
保存残差が小さいことだけで解の精度がよいとは判断できません。

`failed`の場合、`t_reached`は**最後に受理した時刻**です。
`failure_trial_end`は失敗した試行ステップの終端時刻で、中間段階での失敗なら実際の検出段階は`failure_reason`に表示されます。
失敗した計算の最終時刻誤差は計算しません。

In [ ]:
@dataclass
class RunResult:
    problem: Problem
    solver: str
    n: int
    order: int
    gamma: float
    cfl: float
    limiter: str
    wave_speeds: str
    roe_entropy: float
    edges: np.ndarray
    x: np.ndarray
    U0: np.ndarray
    U: np.ndarray
    t: float
    target_time: float
    steps: int
    elapsed: float
    status: str
    initial_total: np.ndarray
    boundary_exchange: np.ndarray
    balance: np.ndarray
    history: np.ndarray
    fail_time: float = np.nan
    fail_dt: float = np.nan
    fail_min_rho: float = np.nan
    fail_min_p: float = np.nan
    fail_reason: str = ""

    @property
    def reconstruction_order(self):
        """The reconstruction option passed to simulate; LW uses constant states."""
        return self.order

    @property
    def space_order_nominal(self):
        """Smooth-solution spatial order, including the coupled Richtmyer update."""
        return 2 if self.solver == "LW" else self.order

    @property
    def time_order(self):
        return 2 if self.solver == "LW" else self.order

    @property
    def label(self):
        name = "LLF (Rusanov)" if self.solver == "LLF" else ("HLL (HLLE)" if self.solver == "HLL" and self.wave_speeds == "einfeldt" else self.solver)
        if self.solver == "LW":
            return "LW (Richtmyer)"
        if self.order == 2:
            return name + " + MUSCL"
        return name


def simulate(problem: Problem, solver: str, n: int = 200, gamma: float = 1.4,
             cfl: float = 0.4, order: int = 1, limiter: str = "minmod",
             wave_speeds: str = "einfeldt", roe_entropy: float = 0.1,
             t_end: Optional[float] = None, max_steps: int = 200000, history_stride: int = 10, observer=None):
    """失敗した試行は棄却して停止する。dt再試行・floor・他ソルバーへの切替は行わない。"""
    if solver not in ALL_SOLVERS or order not in (1, 2):
        raise ValueError("Invalid solver or order.")
    if order == 2 and solver not in RIEMANN_SOLVERS:
        raise ValueError("MUSCL+SSP-RK2 is used only with Roe, LLF, HLL, and HLLC.")
    if not np.isfinite([cfl, gamma, roe_entropy]).all() or not (0.0 < cfl <= 0.9) or gamma <= 1.0 or roe_entropy < 0.0:
        raise ValueError("Require finite 0 < CFL <= 0.9, gamma > 1 and entropy parameter >= 0.")
    if any(isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 1 for k in (history_stride, max_steps)):
        raise ValueError("history_stride and max_steps must be positive integers.")
    target = problem.final_time if t_end is None else float(t_end)
    if not np.isfinite(target) or target < 0.0:
        raise ValueError("t_end must be finite and nonnegative.")
    edges, x, dx = grid_for(problem, n)
    U0 = initial_cell_average(problem, edges, gamma)
    U = U0.copy()
    initial_total = dx * np.sum(U0, axis=0)
    exchanged = np.zeros(3)
    W0 = cons_to_prim(U0, gamma)
    history = [[0.0, float(W0[:, 0].min()), float(W0[:, 2].min()), 0.0]]
    t, steps, status = 0.0, 0, "ok"
    failure = {}
    start = time.perf_counter()
    if observer is not None:
        observer(t, U)  # 表示用の記録だけを行う。Uを変更しない。
    while t < target:
        if steps >= max_steps:
            status = "max_steps"
            failure["fail_reason"] = "Maximum step count reached."
            break
        dt_cfl = cfl * dx / max_signal_speed(U, gamma)
        remaining = target - t
        # LFでは極端に小さい最終dtが過剰な拡散を生むため、最後の2ステップを均等にする。
        dt = dt_cfl if remaining > 2.0 * dt_cfl else remaining / max(1, int(np.ceil(remaining / dt_cfl)))
        if not np.isfinite(dt) or dt <= 0.0 or t + dt == t:
            status = "failed"
            failure = {"fail_reason": "The time step cannot advance floating-point time.", "fail_time": t, "fail_dt": dt}
            break
        try:
            with np.errstate(over="raise", divide="raise", invalid="raise"):
                trial, net_inflow = advance(U, problem, solver, dt, dx, gamma, order, limiter, wave_speeds, roe_entropy)
        except NumericalFailure as error:
            status = "failed"
            failure = {"fail_time": t + dt, "fail_dt": dt, "fail_min_rho": error.min_rho, "fail_min_p": error.min_p, "fail_reason": str(error)}
            break
        except FloatingPointError as error:
            status = "failed"
            failure = {"fail_time": t + dt, "fail_dt": dt, "fail_reason": "Floating-point failure: " + str(error)}
            break
        U = trial
        exchanged += net_inflow
        t = min(target, t + dt)
        steps += 1
        if observer is not None:
            observer(t, U)
        if steps % history_stride == 0 or t == target:
            W = cons_to_prim(U, gamma)
            balance = dx * np.sum(U, axis=0) - initial_total - exchanged
            history.append([t, float(W[:, 0].min()), float(W[:, 2].min()), float(np.max(np.abs(balance)))])
    elapsed = time.perf_counter() - start
    balance = dx * np.sum(U, axis=0) - initial_total - exchanged
    if history[-1][0] != t:
        W = cons_to_prim(U, gamma)
        history.append([t, float(W[:, 0].min()), float(W[:, 2].min()), float(np.max(np.abs(balance)))])
    return RunResult(problem, solver, n, order, gamma, cfl, limiter, wave_speeds, roe_entropy, edges, x, U0, U, t, target, steps, elapsed, status, initial_total, exchanged, balance, np.asarray(history), **failure)

## 7. 評価指標と表示の準備

密度の誤差は、領域長$L$で規格化した平均$L_1$誤差

$$
\epsilon_1(\rho)=\frac1L\sum_i\Delta x\,|\overline\rho_i-\overline\rho_{i,{\rm exact}}|
$$

です。滑らかな周期波については、基本Fourierモードの振幅比`amp_ratio`と位相誤差`phase_rad`も測定します。
接触面の`rho_overshoot`は$\max(\rho)-2$の正の部分、`rho_undershoot`は$1-\min(\rho)$の正の部分です。
`mixed_length`は$1.1<\rho<1.9$のセル幅の合計で、二つの遷移領域と振動の影響を含む補助指標です。単一の接触面の厳密な10–90%幅ではありません。

図は岡部・伊藤パレット、英語ラベル、線種とマーカーの併用に統一します。
RoeとHLLCがほぼ重なること自体も、このテストでは重要な結果です。

In [ ]:
def validate_reference(reference: RunResult, problem: Problem, gamma: float, t: float):
    """Require the same physical problem, gas and time, independent of solver/grid.

    title and final_time are display/default-run settings; the actual comparison
    time is checked separately. All initial/boundary-condition fields must match.
    """
    fields = ("key", "bounds", "boundary", "left", "right", "x0", "velocity")
    if any(not np.array_equal(getattr(reference.problem, name), getattr(problem, name)) for name in fields):
        raise ValueError("Reference and numerical solutions must describe the same physical problem (initial and boundary conditions).")
    if (reference.status != "ok" or not np.isfinite([reference.t, t, reference.gamma, gamma]).all()
            or abs(reference.t - t) > 1.0e-12 or reference.gamma != gamma):
        raise ValueError("The reference must be complete at the same time and gamma as the numerical solution.")


def project_reference(reference: RunResult, edges: np.ndarray) -> np.ndarray:
    """同じ一様格子領域で整数比の細格子の保存変数を粗格子へ体積平均する。"""
    n = len(edges) - 1
    if reference.status != "ok" or reference.n % n or not np.allclose(reference.edges[[0, -1]], edges[[0, -1]], rtol=0.0, atol=1.0e-13):
        raise ValueError("The completed reference must cover the same domain with an integer grid ratio.")
    return reference.U.reshape(n, reference.n // n, 3).mean(axis=1)


def metrics(result: RunResult, reference: Optional[RunResult] = None) -> dict:
    if reference is not None and result.problem.key == "shu_osher":
        validate_reference(reference, result.problem, result.gamma,
                           result.t if result.status == "ok" else result.target_time)
    W = cons_to_prim(result.U, result.gamma)
    row = {"problem": result.problem.key, "solver": result.solver, "order": result.space_order_nominal,
           "reconstruction_order": result.reconstruction_order, "space_order_nominal": result.space_order_nominal,
           "time_order": result.time_order,
           "n": result.n, "gamma": result.gamma, "cfl": result.cfl, "limiter": result.limiter if result.order == 2 else "constant",
           "wave_speeds": result.wave_speeds, "roe_entropy": result.roe_entropy,
           "status": result.status, "t_reached": result.t, "t_target": result.target_time, "steps": result.steps,
           "seconds": result.elapsed, "rho_min": float(W[:, 0].min()), "p_min": float(W[:, 2].min()),
           "balance_max": float(np.max(np.abs(result.balance))), "mass_balance": result.balance[0],
           "momentum_balance": result.balance[1], "energy_balance": result.balance[2],
           "L1_rho": np.nan, "L1_u": np.nan, "L1_p": np.nan, "amp_ratio": np.nan, "phase_rad": np.nan,
           "rho_overshoot": np.nan, "rho_undershoot": np.nan, "mixed_length": np.nan,
           "failure_trial_end": result.fail_time, "failure_dt": result.fail_dt, "failure_rho_min": result.fail_min_rho,
           "failure_p_min": result.fail_min_p, "failure_reason": result.fail_reason}
    if result.status != "ok":
        return row
    Uexact = None
    if result.problem.key != "shu_osher":
        Uexact = exact_cell_average(result.problem, result.edges, result.t, result.gamma)
    elif reference is not None:
        Uexact = project_reference(reference, result.edges)
    if Uexact is not None:
        Wexact = cons_to_prim(Uexact, result.gamma)
        error = np.mean(np.abs(W - Wexact), axis=0)
        row.update(dict(zip(("L1_rho", "L1_u", "L1_p"), error)))
    if result.problem.key == "entropy":
        origin, end = result.problem.bounds
        phase = np.exp(-2j * np.pi * (result.x - origin) / (end - origin))
        coefficient = np.sum((W[:, 0] - W[:, 0].mean()) * phase)
        exact_coefficient = np.sum((Wexact[:, 0] - Wexact[:, 0].mean()) * phase)
        ratio = coefficient / exact_coefficient
        row["amp_ratio"], row["phase_rad"] = float(abs(ratio)), float(np.angle(ratio))
    if result.problem.key in ("contact", "contact_static"):
        row["rho_overshoot"] = float(max(0.0, W[:, 0].max() - 2.0))
        row["rho_undershoot"] = float(max(0.0, 1.0 - W[:, 0].min()))
        row["mixed_length"] = float(np.sum((W[:, 0] > 1.1) & (W[:, 0] < 1.9)) * (result.edges[1] - result.edges[0]))
    return row


def shock_location_error(result: RunResult):
    """右向き衝撃波の近傍で最大密度勾配の位置を比較する簡易指標。"""
    if result.status != "ok" or result.problem.right is None:
        return np.nan
    data = riemann_structure(tuple(result.problem.left), tuple(result.problem.right), result.gamma)
    if data["pstar"] <= result.problem.right[2]:
        return np.nan
    expected = result.problem.x0 + result.t * data["right_speeds"][0]
    rho = result.U[:, 0]
    face_x = 0.5 * (result.x[:-1] + result.x[1:])
    dx = result.edges[1] - result.edges[0]
    window = np.abs(face_x - expected) < max(0.06 * (result.edges[-1] - result.edges[0]), 4.0 * dx)
    indices = np.flatnonzero(window)
    if not len(indices):
        return np.nan
    index = indices[np.argmax(np.abs(np.diff(rho)[indices]))]
    return float(face_x[index] - expected)

In [ ]:
import csv
import html
import json
import platform
import sys
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# 岡部・伊藤パレット。方式ごとの色はすべての図で共通にする。
COLORS = {"LF": "#0072B2", "LW": "#D55E00", "Roe": "#009E73", "LLF": "#CC79A7", "HLL": "#E69F00", "HLLC": "#56B4E9"}
LINES = {"LF": "-", "LW": "-", "Roe": "--", "LLF": "-.", "HLL": ":", "HLLC": "--"}
MARKERS = {"LF": "s", "LW": "^", "Roe": "o", "LLF": "D", "HLL": "v", "HLLC": "x"}
plt.rcParams.update({"figure.dpi": 115, "font.size": 10, "axes.labelsize": 11, "axes.titlesize": 11, "legend.fontsize": 8.5, "mathtext.fontset": "dejavusans"})


def show_table(rows, columns=None, digits=4):
    """pandasを要求しない簡単なHTML表。データ数が多い表も横スクロールできる。"""
    if not rows:
        print("No rows.")
        return
    columns = list(rows[0]) if columns is None else list(columns)
    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return "—" if not np.isfinite(value) else f"{value:.{digits}g}"
        return str(value)
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(format_value(row.get(c, '')))}</td>" for c in columns) + "</tr>" for row in rows)
    style = "<style>.euler-table{border-collapse:collapse;font-size:12px}.euler-table th,.euler-table td{padding:5px 9px;border-bottom:1px solid #ccc;text-align:right}.euler-table th{background:#eee;color:#111}</style>"
    display(HTML(style + '<div style="overflow-x:auto"><table class="euler-table"><thead><tr>' + header + "</tr></thead><tbody>" + body + "</tbody></table></div>"))


def print_failures(results):
    for run in results:
        if run.status != "ok":
            print(f"{run.label}: stopped at last accepted t={run.t:.8g}; rejected trial ended at t={run.fail_time:.8g}.")
            print("  " + run.fail_reason)


def field_values(W, field, gamma):
    if field == "rho":
        return W[..., 0], r"Density $\rho$"
    if field == "u":
        return W[..., 1], r"Velocity $u$"
    if field == "p":
        return W[..., 2], r"Pressure $p$"
    if field == "e":
        return W[..., 2] / ((gamma - 1.0) * W[..., 0]), r"Specific internal energy $e$"
    raise ValueError("Choose rho, u, p, or e.")


def plot_profiles(results, fields=("rho",), reference=None, xlim=None, logarithmic=False, title_suffix=""):
    """各物理量を独立した図に描く。失敗した計算を最終時刻の比較図に混ぜない。"""
    complete = [r for r in results if r.status == "ok"]
    if not complete:
        print("No completed run to plot.")
        print_failures(results)
        return
    base = complete[0]
    if any(abs(r.t - base.t) > 1.0e-12 or r.problem != base.problem or r.gamma != base.gamma for r in complete):
        raise ValueError("All plotted solutions must have the same problem, time, and gamma.")
    if reference is not None and base.problem.key == "shu_osher":
        validate_reference(reference, base.problem, base.gamma, base.t)
    for field in fields:
        fig = plt.figure(figsize=(9.8, 4.5))
        ax = fig.gca()
        if base.problem.key != "shu_osher":
            xp = np.linspace(*base.problem.bounds, 2401)
            Wp = exact_pointwise(base.problem, xp, base.t, base.gamma)
            yp, ylabel = field_values(Wp, field, base.gamma)
            ax.plot(xp, yp, color="#000000", lw=1.3, label="Exact (pointwise)", zorder=5)
        elif reference is not None:
            yp, ylabel = field_values(cons_to_prim(reference.U, reference.gamma), field, reference.gamma)
            ax.plot(reference.x, yp, color="#000000", lw=1.2, label=f"Numerical ref. N={reference.n}", zorder=5)
        for index, run in enumerate(complete):
            y, ylabel = field_values(cons_to_prim(run.U, run.gamma), field, run.gamma)
            stride = max(1, run.n // 22)
            ax.plot(run.x, y, color=COLORS[run.solver], ls=LINES[run.solver], lw=1.25, marker=MARKERS[run.solver], ms=3.1, markevery=(index % stride, stride), mfc="none", label=run.label)
        ax.set(xlabel="$x$", ylabel=ylabel, xlim=base.problem.bounds if xlim is None else xlim)
        ax.set_title(f"{base.problem.title}: t={base.t:g}, N={base.n}" + ("\n" + title_suffix if title_suffix else ""))
        if logarithmic:
            ax.set_yscale("log")
        ax.grid(alpha=0.22)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    print_failures(results)


def plot_convergence(results, title):
    fig = plt.figure(figsize=(9.5, 4.7))
    ax = fig.gca()
    for solver in ALL_SOLVERS:
        selected = sorted([r for r in results if r.solver == solver and r.status == "ok"], key=lambda r: r.n)
        if not selected:
            continue
        n = np.array([r.n for r in selected])
        error = np.array([metrics(r)["L1_rho"] for r in selected])
        ax.loglog(n, error, color=COLORS[solver], marker=MARKERS[solver], ls=LINES[solver], ms=5, mfc="none", label=selected[0].label)
    ax.set(xlabel="Number of cells $N$", ylabel=r"Mean $L_1$ error in density", title=title)
    ax.grid(which="both", alpha=0.22)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def convergence_rows(results):
    rows = []
    for solver in ALL_SOLVERS:
        subset = sorted([r for r in results if r.solver == solver], key=lambda r: r.n)
        previous = None
        for r in subset:
            row = metrics(r)
            row["q_observed"] = np.nan if previous is None or row["status"] != "ok" else np.log(previous["L1_rho"] / row["L1_rho"]) / np.log(row["n"] / previous["n"])
            rows.append(row)
            previous = row
    return rows


def plot_pressure_minimum(results):
    fig = plt.figure(figsize=(9.5, 4.7))
    ax = fig.gca()
    for run in results:
        ax.plot(run.history[:, 0], run.history[:, 2], color=COLORS[run.solver], ls=LINES[run.solver], label=run.label)
        if np.isfinite(run.fail_min_p):
            ax.plot(run.fail_time, run.fail_min_p, marker="x", ms=9, color=COLORS[run.solver], ls="none")
    ax.axhline(0.0, color="#000000", lw=0.8)
    ax.set_yscale("symlog", linthresh=1.0e-3)
    ax.set(xlabel="Time", ylabel=r"Minimum pressure $p_{\min}$", title="Pressure positivity: lines = accepted states; crosses = rejected trials")
    ax.grid(alpha=0.22)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
"""Track experiment membership independently of numerical-result reuse."""
from dataclasses import asdict
from pathlib import Path
import json
import numpy as np


def json_value(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")


class ExperimentCache(dict):
    """A result may be reused by several experiments, identified by cfg snapshots."""
    def __init__(self):
        super().__init__()
        self._experiments = {}

    def record(self, cfg, key):
        signature = json.dumps(cfg, sort_keys=True, default=json_value)
        self._experiments.setdefault(signature, {})[key] = None

    def runs_for(self, cfg):
        signature = json.dumps(cfg, sort_keys=True, default=json_value)
        return [self[key] for key in self._experiments.get(signature, {}) if key in self]

    def clear(self):
        super().clear()
        self._experiments.clear()


def run_settings(run):
    """Physical and numerical settings for one saved array group."""
    settings = {name: getattr(run, name) for name in
                ("solver", "n", "reconstruction_order", "space_order_nominal", "time_order",
                 "gamma", "cfl", "limiter", "wave_speeds", "roe_entropy", "target_time")}
    settings["order"] = run.space_order_nominal  # Legacy output alias, never a simulate argument.
    settings["problem"] = asdict(run.problem)
    for name in ("reconstruction", "time_integrator", "fixed_steps"):
        if hasattr(run, name):
            settings[name] = getattr(run, name)
    return settings


def run_metadata(run_id, prefix, run, reference=None):
    if run.status != "ok":
        error_reference = "none: incomplete"
    elif run.problem.key != "shu_osher":
        error_reference = "exact cell averages"
    else:
        error_reference = "numerical" if reference is not None else "none"
    return dict(run_id=run_id, array_prefix=prefix, settings=run_settings(run),
                status=run.status, t_reached=run.t, steps=run.steps,
                error_reference=error_reference,
                reference_settings=run_settings(reference) if reference is not None else None)


RUN_CACHE = ExperimentCache()


def get_run(key, solver, n=None, order=1, **overrides):
    problem = PROBLEMS[key] if isinstance(key, str) else key
    parameters = dict(n=CFG["n_basic"] if n is None else n, gamma=CFG["gamma"], cfl=CFG["cfl"], order=order,
                      limiter=CFG["limiter"], wave_speeds=CFG["wave_speeds"], roe_entropy=CFG["roe_entropy"])
    parameters.update(overrides)
    cache_key = (problem, solver, tuple(sorted(parameters.items())))
    if cache_key not in RUN_CACHE:
        RUN_CACHE[cache_key] = simulate(problem, solver, **parameters)
    RUN_CACHE.record(CFG, cache_key)
    return RUN_CACHE[cache_key]


def run_suite(key, n=None, order=1, solvers=None):
    solvers = (ALL_SOLVERS if order == 1 else RIEMANN_SOLVERS) if solvers is None else solvers
    results = [get_run(key, solver, n=n, order=order) for solver in solvers]
    print(f"{key}: " + ", ".join(f"{r.solver}={r.status}" for r in results))
    return results


def summarize(results, reference=None, columns=None):
    columns = ("solver", "n", "order", "status", "t_reached", "L1_rho", "L1_u", "L1_p", "rho_min", "p_min", "balance_max", "seconds") if columns is None else columns
    rows = [metrics(r, reference=reference) for r in results]
    show_table(rows, columns)
    return rows

## 8. 実装の自己検証

変数変換、数値流束の整合性$F^*(U,U)=F(U)$、対称波速HLLとLLFの一致、静止接触面の保持、周期系の保存、非周期境界の収支を検証します。
さらにSodの$p_*,u_*$と、厳密解の12点・24点Gauss積分の一致を確認します。
これは実装の基本チェックであり、任意の初期条件に対する正しさの証明ではありません。

In [ ]:
def run_unit_tests(gamma: float = 1.4):
    """数値流束、厳密解、境界の保存則を小さい計算で検証する。失敗時はassertで止まる。"""
    tests = []
    rng = np.random.default_rng(7341)
    W = np.column_stack((np.exp(rng.uniform(-1.0, 1.0, 31)), rng.uniform(-0.8, 0.8, 31), np.exp(rng.uniform(-1.0, 1.0, 31))))
    U = prim_to_cons(W, gamma)
    np.testing.assert_allclose(cons_to_prim(U, gamma), W, rtol=1.0e-13, atol=1.0e-13)
    tests.append("primitive/conserved round trip")
    for solver in ALL_SOLVERS:
        F = numerical_flux(U, U, solver, gamma, dt=0.01, dx=0.1)
        np.testing.assert_allclose(F, physical_flux(U, gamma), rtol=2.0e-13, atol=2.0e-13)
    tests.append("F*(U,U) = F(U), all six solvers")
    WL, WR = W[:15], W[15:30]
    UL, UR = prim_to_cons(WL, gamma), prim_to_cons(WR, gamma)
    Fhll = numerical_flux(UL, UR, "HLL", gamma, wave_speeds="symmetric")
    Fllf = numerical_flux(UL, UR, "LLF", gamma)
    np.testing.assert_allclose(Fhll, Fllf, rtol=2.0e-13, atol=2.0e-13)
    tests.append("symmetric-speed HLL equals LLF")
    for solver in ("Roe", "HLLC", "LW"):
        run = simulate(PROBLEMS["contact_static"], solver, n=40, gamma=gamma, t_end=0.02)
        assert run.status == "ok"
        np.testing.assert_allclose(run.U, run.U0, atol=3.0e-13, rtol=3.0e-13)
    tests.append("stationary contact: Roe, HLLC, and Richtmyer LW")
    for solver in ALL_SOLVERS:
        run = simulate(PROBLEMS["entropy"], solver, n=40, gamma=gamma, t_end=0.03)
        assert run.status == "ok"
        assert np.max(np.abs(run.balance)) < 1.0e-11
    tests.append("periodic conservation, all six solvers")
    for solver in RIEMANN_SOLVERS:
        run = simulate(PROBLEMS["entropy"], solver, n=40, gamma=gamma, order=2, t_end=0.03)
        assert run.status == "ok"
        assert np.max(np.abs(run.balance)) < 1.0e-11
    tests.append("MUSCL/SSP-RK2 periodic conservation")
    if abs(gamma - 1.4) < 1.0e-14:
        problem = PROBLEMS["sod"]
        data = riemann_structure(tuple(problem.left), tuple(problem.right), gamma)
        np.testing.assert_allclose((data["pstar"], data["ustar"]), (0.303130178050647, 0.927452620048950), rtol=2.0e-11)
        tests.append("Sod star pressure and velocity")
    for key in ("sod", "double_rarefaction", "lax", "transonic"):
        problem = PROBLEMS[key]
        edges, _, _ = grid_for(problem, 50)
        ref12 = exact_cell_average(problem, edges, problem.final_time, gamma, nq=12)
        ref24 = exact_cell_average(problem, edges, problem.final_time, gamma, nq=24)
        np.testing.assert_allclose(ref12, ref24, rtol=2.0e-10, atol=2.0e-12)
        cons_to_prim(ref12, gamma)
    tests.append("exact-solution quadrature and positive reference states")
    # 流入・流出がある場合に、生の総量変化ではなく境界流束を引いた残差を見る。
    run = simulate(PROBLEMS["shu_osher"], "HLLC", n=40, gamma=gamma, order=2, t_end=0.03)
    assert run.status == "ok" and np.max(np.abs(run.balance)) < 1.0e-10
    assert np.max(np.abs(run.boundary_exchange)) > 1.0e-6
    tests.append("nonperiodic boundary-flux budget")
    return tests

In [ ]:
passed = run_unit_tests(CFG["gamma"])
for index, test in enumerate(passed, 1):
    print(f"PASS {index}: {test}")
print(f"All {len(passed)} validation groups passed.")

# Stage 1：基本6方式の比較

## 9. 滑らかなエントロピー波

$u=p=1$のまま密度だけが移流します。$t=1$で1周して初期波形に戻ります。
**どの程度振幅が減るか、位相がずれるか、圧力・速度を一定に保てるか**を見てください。
不連続を含まないので、ここではLWの2次精度が有利になると予想できます。
接触波を分離するRoe・HLLCと、音速に関連する散逸も含むLLF・HLLの違いも観察します。

In [ ]:
entropy_runs = run_suite("entropy")
plot_profiles(entropy_runs)
entropy_rows = summarize(entropy_runs, columns=("solver", "order", "L1_rho", "L1_u", "L1_p", "amp_ratio", "phase_rad", "balance_max", "seconds"))

## 10. 静止する接触不連続面

$u=0, p=1$で密度だけを不連続にします。RoeとHLLCに接触波を扱う構造があることを、最も単純な状態で確認します。
この特別な静止解ではRichtmyer LWも流束を一定に保てます。
**静止接触面の保持だけでは、LWの分散誤差や移動する接触面の性能は判定できません。**

In [ ]:
static_runs = run_suite("contact_static")
plot_profiles(static_runs)
static_rows = summarize(static_runs, columns=("solver", "L1_rho", "rho_overshoot", "rho_undershoot", "mixed_length", "balance_max"))

## 11. 移動する接触不連続面

今度は$u=1$です。$t=0.25$では密度2の領域が$0.5\le x<0.7$に移動します。
矩形波の境界の広がりと、範囲$1\le\rho\le2$からの逸脱を比較します。
**接触波を分離できるRoe/HLLCでも、移動する不連続面を1次精度で離散化すれば数値拡散は残ります。**

In [ ]:
contact_runs = run_suite("contact")
plot_profiles(contact_runs)
plot_profiles(contact_runs, xlim=(0.58, 0.82), title_suffix="Zoom near the right contact")
contact_rows = summarize(contact_runs, columns=("solver", "L1_rho", "rho_overshoot", "rho_undershoot", "mixed_length", "balance_max"))

## 12. Sod衝撃波管：三つの波を分けて読む

左から希薄波、接触不連続面、衝撃波が現れます。密度だけでなく速度・圧力も必ず見てください。
圧力・速度が連続のまま密度だけが跳ぶ位置が接触面です。

衝撃波位置の追加指標`shock_error`は、厳密な衝撃波位置の近傍で密度差分が最大の場所を探した簡易評価です。
振動や粗い格子に影響されるため、サブセル精度の波面推定としては使用しません。

In [ ]:
sod_runs = run_suite("sod")
plot_profiles(sod_runs, fields=("rho", "u", "p"))
sod_rows = [dict(metrics(r), shock_error=shock_location_error(r)) for r in sod_runs]
show_table(sod_rows, ("solver", "status", "L1_rho", "L1_u", "L1_p", "shock_error", "balance_max", "seconds"))
data = riemann_structure(PROBLEMS["sod"].left, PROBLEMS["sod"].right, CFG["gamma"])
print(f"Exact p* = {data['pstar']:.12f}; u* = {data['ustar']:.12f}")
print(f"Contact x = {0.5 + 0.2 * data['ustar']:.8f}; right shock x = {0.5 + 0.2 * data['right_speeds'][0]:.8f}")

## 13. 二重希薄波：低密度・低圧力と計算の破綻

左右のガスが反対方向に離れ、中央で密度・圧力が小さくなります。
正値性に問題のある方式を、Sodだけでは検出できない場合があります。[R2]

以下では完走した解だけを最終時刻に比較します。
負の圧力などで停止した計算は表と圧力最小値の履歴に残します。
履歴図の実線は受理した状態、×印は棄却した試行の最小圧力です。
対数軸では負圧を表示できないため、履歴図のみゼロ近傍が線形になる`symlog`を用います。

内部エネルギー$e$も描きます。中央で$e$に誤差が残ることと、計算が負圧で破綻することは異なる問題です。

In [ ]:
rarefaction_runs = run_suite("double_rarefaction")
plot_profiles(rarefaction_runs, fields=("rho", "p"), logarithmic=True)
plot_profiles(rarefaction_runs, fields=("e",))
rarefaction_rows = summarize(rarefaction_runs, columns=("solver", "status", "t_reached", "failure_trial_end", "failure_p_min", "L1_rho", "rho_min", "p_min", "balance_max"))
plot_pressure_minimum(rarefaction_runs)
data = riemann_structure(PROBLEMS["double_rarefaction"].left, PROBLEMS["double_rarefaction"].right, CFG["gamma"])
print(f"Exact central density = {data['rhoLstar']:.10g}; pressure = {data['pstar']:.10g}")

## 14. 格子収束試験：滑らかな解で精度を測る

格子数$N_1,N_2$の誤差を使い、

$$q_{\rm obs}=\frac{\log[\epsilon_1(N_1)/\epsilon_1(N_2)]}{\log(N_2/N_1)}$$

を測定します。Stage 1ではLWだけが形式的に2次で、他は1次です。
LFのように粗い格子で波が大きく減衰する場合、まだ漸近的な収束領域に入らず、$q_{\rm obs}$が1よりかなり小さくなることがあります。

衝撃波・接触不連続面を含む全領域の誤差から、そのまま滑らかな解に対する形式的精度を判定しないでください。

In [ ]:
convergence_stage1 = []
if CFG["run_convergence"]:
    convergence_stage1 = [get_run("entropy", s, n=n) for s in ALL_SOLVERS for n in CFG["n_convergence"]]
    plot_convergence(convergence_stage1, "Stage 1: smooth entropy-wave convergence")
    show_table(convergence_rows(convergence_stage1), ("solver", "order", "n", "L1_rho", "q_observed", "amp_ratio", "phase_rad"))
else:
    print("Convergence tests skipped.")

# Stage 2：MUSCL＋SSP-RK2で条件を統一する

## 15. 移動接触面とSod問題の再計算

同じ基本変数、同じリミッター、同じSSP-RK2を使い、数値流束だけを切り替えます。
ここではLFとLWを含めず、Roe・LLF・HLL・HLLCを比較します。
Stage 1と見比べて、流束の選択と再構成の変更のどちらが効いているかを調べてください。

In [ ]:
contact_muscl = run_suite("contact", order=2)
plot_profiles(contact_muscl, xlim=(0.42, 0.8))
contact_muscl_rows = summarize(contact_muscl, columns=("solver", "order", "L1_rho", "rho_overshoot", "rho_undershoot", "mixed_length"))

sod_muscl = run_suite("sod", order=2)
plot_profiles(sod_muscl, fields=("rho", "p"))
sod_muscl_rows = summarize(sod_muscl)

### 15.1 高次精度化後の収束次数

minmodは正弦波の極大・極小付近でも再構成を制限するため、有限の格子数では$L_1$誤差の次数が2より少し低い場合があります。
`limiter="unlimited"`をこの滑らかな問題だけで試すと、リミッターの影響を切り分けられます。

In [ ]:
convergence_stage2 = []
if CFG["run_convergence"]:
    convergence_stage2 = [get_run("entropy", s, n=n, order=2) for s in RIEMANN_SOLVERS for n in CFG["n_convergence"]]
    plot_convergence(convergence_stage2, "Stage 2: MUSCL + SSP-RK2, smooth entropy wave")
    show_table(convergence_rows(convergence_stage2), ("solver", "order", "n", "limiter", "L1_rho", "q_observed"))

### 15.2 二重希薄波を再確認する

高次精度化しただけで低密度問題の頑健性が改善するとは限りません。
再構成・中間RK段階・最終状態をそれぞれ検査し、どの段階で失敗するかも記録します。

In [ ]:
rarefaction_muscl = run_suite("double_rarefaction", order=2)
rarefaction_muscl_rows = summarize(rarefaction_muscl, columns=("solver", "order", "status", "t_reached", "failure_trial_end", "failure_p_min", "L1_rho", "rho_min", "p_min"))
print_failures(rarefaction_muscl)

## 16. Shu–Osher問題：衝撃波と短波長構造を同時に解く

衝撃波が正弦波状の密度変動へ進み、通過後の密度に細かな構造を作ります。
衝撃波近傍の非物理的な振動と、衝撃波後方に物理的に存在する振動を区別する必要があります。[R5, R9]

以下では、同じHLLC＋MUSCLで**2段階の高解像度参照解**を計算します。
参照解のリミッターは`reference_limiter`で指定し、標準はMCです。
小さい$N$の参照解と大きい$N$の参照解の差も表示します。

粗格子との誤差評価には、細格子の**保存変数を体積平均して**粗格子へ射影します。
衝撃波を含む密度・圧力の点値を、単純な線形補間で粗格子へ持ってくる方法は使いません。
したがって、参照解の格子数は比較用格子数の整数倍にしてください。

参照解は厳密解ではなく、同じソルバー系列による誤差・バイアスが残ります。
参照解同士の差と同程度以下の誤差で方式の順位が変わる場合は、さらに細かい格子や別系列の参照解で検証してください。

標準の400セル・minmodでは短波長構造の振幅が大きく減衰します。`n_shu_osher=800`などに増やして比較してください。参照解同士の差は、参照解の誤差の厳密な上限ではありません。

In [ ]:
shu_stage1, shu_stage2 = [], []
reference_coarse, reference_fine = None, None
reference_checks = []
if CFG["run_shu_osher"]:
    nref1, nref2 = CFG["n_reference"]
    if nref2 <= nref1 or nref2 % nref1 or nref1 % CFG["n_shu_osher"]:
        raise ValueError("Choose nested integer grid ratios: comparison N -> reference N1 -> reference N2.")
    reference_coarse = get_run("shu_osher", "HLLC", n=nref1, order=2, limiter=CFG["reference_limiter"])
    if reference_coarse.status != "ok":
        raise RuntimeError("The coarse reference failed.")
    print(f"First numerical reference completed: N={nref1}.")
else:
    print("Shu-Osher tests skipped.")


In [ ]:
if CFG["run_shu_osher"]:
    reference_fine = get_run("shu_osher", "HLLC", n=nref2, order=2, limiter=CFG["reference_limiter"])
    if reference_fine.status != "ok":
        raise RuntimeError("The fine reference failed; do not use it for comparison.")
    Ufine_projected = project_reference(reference_fine, reference_coarse.edges)
    W1, W2 = cons_to_prim(reference_coarse.U, CFG["gamma"]), cons_to_prim(Ufine_projected, CFG["gamma"])
    downstream = (reference_coarse.x > -1.0) & (reference_coarse.x < 2.0)
    reference_checks = [
        {"region": "full domain", "N1": nref1, "N2": nref2, "mean_abs_density_difference": np.mean(np.abs(W1[:, 0] - W2[:, 0]))},
        {"region": "-1 < x < 2", "N1": nref1, "N2": nref2, "mean_abs_density_difference": np.mean(np.abs(W1[downstream, 0] - W2[downstream, 0]))},
    ]
    show_table(reference_checks)
    print(f"References completed: HLLC + MUSCL ({CFG['reference_limiter']}), N={nref1} and {nref2}.")


In [ ]:
if CFG["run_shu_osher"]:
    shu_stage1 = run_suite("shu_osher", n=CFG["n_shu_osher"])
    shu_stage2 = run_suite("shu_osher", n=CFG["n_shu_osher"], order=2)
    plot_profiles(shu_stage1, reference=reference_fine, title_suffix="Stage 1")
    plot_profiles(shu_stage2, reference=reference_fine, title_suffix="Stage 2")
    plot_profiles(shu_stage2, reference=reference_fine, xlim=(-1.0, 2.3), title_suffix="Zoom: physical density oscillations behind the shock")
    shu_rows = [metrics(r, reference_fine) for r in shu_stage1 + shu_stage2]
    show_table(shu_rows, ("solver", "order", "n", "status", "L1_rho", "L1_u", "L1_p", "balance_max", "seconds"))
    print("These L1 values are differences from a numerical reference, not errors against an exact solution.")

# 追加実験

## 17. CFL数依存性：LFとLLF

線形移流で定数係数を仮定した修正方程式から、代表的な数値粘性係数は

$$
\nu_{\rm LF}\simeq\frac{\Delta x^2}{2\Delta t}-\frac{u^2\Delta t}{2},\qquad
\nu_{\rm LLF}\simeq\frac{\alpha\Delta x}{2}-\frac{u^2\Delta t}{2}
$$

となります。LFでは$\Delta t$を小さくするほど主な散逸係数が増します。
「時間刻みを小さくすれば必ず精度が上がる」とは言えない例です。
以下のEuler計算では局所波速と時間刻みが変化するため、この式は機構を理解するための近似として読んでください。

In [ ]:
cfl_runs = []
if CFG["run_extras"]:
    cfl_runs = [get_run("entropy", s, cfl=c) for s in ("LF", "LLF") for c in (0.2, 0.4, 0.8)]
    show_table([metrics(r) for r in cfl_runs], ("solver", "cfl", "n", "L1_rho", "amp_ratio", "phase_rad", "steps", "seconds"))

## 18. 音速をまたぐ希薄波：Roeのエントロピー補正

変更したSod型初期条件では、左希薄波の$u-c$が符号を変えます。
Roeの補正なし・ありを同じ格子数で比較します。
この実験が調べるのは**非物理的な膨張衝撃波の抑制**であり、二重希薄波の正値性問題とは分けて考えます。

In [ ]:
transonic_runs = []
if CFG["run_extras"]:
    transonic_runs = [get_run("transonic", "Roe", n=400, roe_entropy=value) for value in (0.0, 0.1)]
    problem = PROBLEMS["transonic"]
    fig = plt.figure(figsize=(9.5, 4.6))
    ax = fig.gca()
    xp = np.linspace(0.19, 0.4, 1201)
    ax.plot(xp, exact_pointwise(problem, xp, problem.final_time, CFG["gamma"])[:, 0], color="#000000", lw=1.3, label="Exact")
    for run, linestyle, marker in zip(transonic_runs, ("--", "-"), ("o", "x")):
        if run.status == "ok":
            ax.plot(run.x, run.U[:, 0], color=COLORS["Roe"], ls=linestyle, marker=marker, ms=3, mfc="none", markevery=3, label=f"Roe, entropy epsilon={run.roe_entropy:g}")
    ax.set(xlim=(0.19, 0.4), xlabel="$x$", ylabel=r"Density $\rho$", title="Transonic rarefaction: effect of the entropy correction")
    ax.grid(alpha=0.22)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    show_table([metrics(r) for r in transonic_runs], ("solver", "roe_entropy", "status", "L1_rho", "L1_p", "p_min"))

## 19. Lax衝撃波管

Laxという名称は初期値問題の名前で、Lax–Friedrichs法やLax–Wendroff法だけを使うという意味ではありません。
Sodと異なる速度・圧力差で、低次精度の拡散やLWの振動を再確認します。

In [ ]:
lax_runs = []
if CFG["run_extras"]:
    lax_runs = run_suite("lax")
    plot_profiles(lax_runs, fields=("rho",))
    lax_rows = summarize(lax_runs)

## 20. 結果を保存する

`euler1d_outputs`に、全計算の設定・誤差・失敗情報を含むCSV、環境情報JSON、状態と履歴のNPZを保存します。
同じ条件の計算はメモリ内キャッシュで再利用しています。

`seconds`は初期条件生成・誤差評価・作図を除く積分ループの壁時計時間です。
1回の測定値であり、Python/NumPy実装の構成にも依存します。
スキーム固有の普遍的な計算コストや、同じ精度を達成するための効率とは区別してください。
厳密な性能比較には、反復実行、中央値、同じ誤差に対する計算時間の評価が必要です。

In [ ]:
selected_runs = RUN_CACHE.runs_for(CFG)
all_rows, saved_metadata = [], []
for i, run in enumerate(selected_runs):
    if reference_fine is not None and run.problem.key == "shu_osher":
        validate_reference(reference_fine, run.problem, run.gamma, run.target_time)
    usable_reference = reference_fine if run.problem.key == "shu_osher" and run is not reference_fine else None
    row = metrics(run, reference=usable_reference)
    row["error_reference"] = "none: failed/incomplete" if run.status != "ok" else ("exact cell averages" if run.problem.key != "shu_osher" else (f"numerical reference N={reference_fine.n}" if usable_reference is not None else "reference run itself"))
    row["run_id"] = i
    all_rows.append(row)
    saved_metadata.append(run_metadata(i, f"run_{i:03d}", run, usable_reference))

if CFG["export_results"]:
    if not selected_runs:
        raise ValueError("Run the experiment for the current CFG before exporting.")
    output_dir = Path("euler1d_outputs")
    output_dir.mkdir(exist_ok=True)
    with (output_dir / "metrics.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(all_rows[0]))
        writer.writeheader()
        writer.writerows(all_rows)
    environment = {"python": sys.version, "numpy": np.__version__, "matplotlib": matplotlib.__version__, "platform": platform.platform(), "settings": CFG, "reference_checks": reference_checks, "validation_groups": passed, "history_columns": ["time", "rho_min", "p_min", "max_abs_balance"], "note": "No positivity floors, retries, or solver fallback. Failed runs retain only the last accepted state."}
    (output_dir / "environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_dir / "runs.json").write_text(json.dumps(saved_metadata, ensure_ascii=False, indent=2, default=json_value), encoding="utf-8")
    arrays = {}
    for i, run in enumerate(selected_runs):
        prefix = f"run_{i:03d}"
        arrays[prefix + "_x"] = run.x
        arrays[prefix + "_edges"] = run.edges
        arrays[prefix + "_U0"] = run.U0
        arrays[prefix + "_U"] = run.U
        arrays[prefix + "_history"] = run.history
        arrays[prefix + "_boundary_exchange"] = run.boundary_exchange
        arrays[prefix + "_balance"] = run.balance
    if reference_fine is not None:
        for name in ("x", "edges", "U"):
            arrays["reference_" + name] = getattr(reference_fine, name)
    np.savez_compressed(output_dir / "states.npz", **arrays)
    print(f"Saved {len(all_rows)} distinct runs in {output_dir.resolve()}")
    print("metrics.csv row order corresponds to run_000, run_001, ... in states.npz.")
    print("states.npz can be loaded with np.load(..., allow_pickle=False).")
else:
    print("Export skipped.")

failed_rows = [row for row in all_rows if row["status"] != "ok"]
print(f"Completed: {len(all_rows) - len(failed_rows)}; intentionally recorded failures/incomplete runs: {len(failed_rows)}")
show_table(failed_rows, ("problem", "solver", "order", "n", "t_reached", "failure_trial_end", "failure_p_min", "failure_reason"))

## 21. 読み取りと演習

**拡散と分散を分ける。** 滑らかな波の振幅低下と、移動接触面のovershootを別々の指標で比べてください。
LWの滑らかな波での小さい誤差と、衝撃波近傍の振動は矛盾しません。

**ソルバーと高次精度化を分ける。** まず同じソルバーでStage 1→Stage 2の変化を比べ、次にStage 2内で流束を変えます。
`minmod`と`mc`を切り替え、特にShu–Osherの密度振動がどの程度変わるか調べてください。
MCを使う場合も、4方式の比較には同じリミッターを使います。

**頑健性と正確さを分ける。** 二重希薄波で完走した方式の内部エネルギーを比較してください。
負圧が出ないことは必要な検査ですが、解が十分に正確であることを意味しません。
反対に、破綻した方式にfloorを加えて図だけ出すと、保存則や失敗原因を見落としやすくなります。

**追加課題。** (i) $u=0$と$u=1$の接触面の違いを説明する。
(ii) HLLの外側波速を対称化しLLFとの一致を確認する。
(iii) CFL数を小さくしたときLFの誤差がどう変わるか説明する。
(iv) Shu–Osherの参照格子数を増やし、方式間の差に対して参照解の不確かさが十分小さいか確認する。
(v) 同じ$L_1$誤差を達成する格子数・計算時間で比較し直す。

**適用範囲。** このコードは1次元・一様格子・単一理想気体の学習用です。
重力、冷却、粘性、MHD、相対論、多次元、真空、研究用途の包括的な耐破綻処理は含みません。
高Mach数での内部エネルギーの桁落ちにも特別な対策はしていません。
これらを追加する際には、今回のテストを回帰テストとして残してください。

## 22. 参考文献・実装の確認先

以下は原論文または開発者による公式資料です。Notebookのコードはこれらの数式・考え方を参考に、本教材用に組み立てた実装であり、Clawpack等を呼び出してはいません。

**[R1]** Clawpack, *The Euler equations of gas dynamics*：波の構造と厳密リーマン解。  
https://www.clawpack.org/riemann_book/html/Euler.html

**[R2]** Clawpack, *Approximate solvers for the Euler equations of gas dynamics*：Roe平均、HLLEの波速推定、低密度問題。  
https://www.clawpack.org/riemann_book/html/Euler_approximate.html

**[R3]** Clawpack, *Riemann Solver Package — Euler Equations*：HLLCの接触波速度と中間状態。  
https://www.clawpack.org/pyclaw/rp.html#euler-equations

**[R4]** FLASH User's Guide, *Hydrodynamics Test Problems*, §35.1.1：Sod問題と波面の位置。  
https://flash.rochester.edu/site/flashcode/user_support/flash_ug_devel/node191.html

**[R5]** Clawpack Gallery, *Shu–Osher problem*：初期状態、領域、終了時刻。  
https://www.clawpack.org/gallery/pyclaw/gallery/shocksine.html

**[R6]** Toro, E. F., Spruce, M., & Speares, W. (1994), *Restoration of the contact surface in the HLL-Riemann solver*, Shock Waves, 4, 25–34.  
https://doi.org/10.1007/BF01414629

**[R7]** Harten, A. (1983), *High resolution schemes for hyperbolic conservation laws*, Journal of Computational Physics, 49, 357–393.  
https://doi.org/10.1016/0021-9991(83)90136-5

**[R8]** Shu, C.-W., & Osher, S. (1988), *Efficient implementation of essentially non-oscillatory shock-capturing schemes*, Journal of Computational Physics, 77, 439–471.  
https://doi.org/10.1016/0021-9991(88)90177-5

**[R9]** Shu, C.-W., & Osher, S. (1989), *Efficient implementation of essentially non-oscillatory shock-capturing schemes, II*, Journal of Computational Physics, 83, 32–78.  
https://doi.org/10.1016/0021-9991(89)90222-2

初期条件・波速推定・エントロピー補正・リミッター・CFL数まで含めて、比較の設定を報告することが重要です。
「Roe」や「HLLC」という名前だけでは、実装のすべては決まりません。

<a id="stage-1-animation"></a>
# Stage 1：6方式の時間発展を動画で比較する

**LF・LW・Roe・LLF（Rusanov）・HLL（HLLE）・HLLCを、同じ物理時刻で重ねて表示します。**
滑らかなエントロピー波、静止接触面、移動接触面、Sod、二重希薄波、Shu–Osherに加えて、追加問題のLax衝撃波管も収録しました。

| 問題 | 標準で作成する動画 |
|:--|:--|
| エントロピー波、静止・移動接触面 | 密度 |
| Sod、Lax衝撃波管 | 密度、速度、圧力をそれぞれ別の動画にする |
| 二重希薄波 | 密度、速度、圧力、比内部エネルギーをそれぞれ別の動画にする |
| Shu–Osher | 密度。黒線は高解像度の**数値参照解** |

保存済み動画は各セルの出力で再生できます。動画のシークバーで時刻を移動し、全画面表示にもできます。
`stage1_gallery.html`は、問題・物理量・再生速度を切り替えられる独立した動画一覧です。
**保存済み動画の再生にffmpegは不要です。** 再生成時はNumPy・Matplotlib・Jupyterに加えて、MP4出力用のffmpegを用います。
ffmpegが見つからない場合は、Notebook内のJavaScriptアニメーションに切り替えます。

この追加部分では**数値流束・初期条件・境界条件・CFL時間刻みの選び方を変更していません**。
LWはRichtmyerの2段階・2次精度法、他の5方式はStage 1の1次精度構成です。したがって、同次数のリーマンソルバーだけの性能比較ではありません。


## 動画設定

主に変更するのは`ANIM_CFG`です。`n_frames`は物理時刻のサンプル数、`fps`は1秒当たりの表示フレーム数、`hold_start`と`hold_end`は最初・最後の静止フレーム数です。
全問題で同じ動画長にしているため、問題間で物理時刻の進む速さは異なります。物理時刻は図の上部に表示します。

格子数・CFL数・フレーム数などを変えた場合は、キャッシュの設定を照合して再計算します。描画設定だけの変更では数値積分をやり直しません。
**流束や時間積分のコードそのものを編集した場合**は`reuse_data=False`か`ANIM_CFG["cache_revision"]="my-edit-1"`を設定してください。
動画の関数自体を編集した場合は`reuse_movies=False`にします。

`fields_by_problem`を変更すると、任意の問題で`"rho"`（密度）、`"u"`（速度）、`"p"`（圧力）、`"e"`（比内部エネルギー）の動画を追加できます。


In [ ]:
from pathlib import Path
import numpy as np
ANIM_CFG = {
    "problems": ("entropy", "contact_static", "contact", "sod", "double_rarefaction", "shu_osher", "lax"),
    "n_basic": 200,
    "n_shu_osher": 400,
    "gamma": 1.4,
    "cfl": 0.4,
    "wave_speeds": "einfeldt",
    "roe_entropy": 0.1,
    "n_frames": 181,
    "fps": 20,
    "hold_start": 10,
    "hold_end": 30,
    "dpi": 100,
    "figsize": (12.8, 7.2),
    "n_reference": (1600, 3200),
    "reference_limiter": "mc",
    "fields_by_problem": {
        "entropy": ("rho",),
        "contact_static": ("rho",),
        "contact": ("rho",),
        "sod": ("rho", "u", "p"),
        "double_rarefaction": ("rho", "u", "p", "e"),
        "shu_osher": ("rho",),
        "lax": ("rho", "u", "p"),
    },
    "output_dir": "stage1_animation_outputs",
    "reuse_data": True,
    "reuse_movies": True,
    "save_mp4": True,
    "verify_observer": True,
}

BASE_DIR = Path.cwd()
if (BASE_DIR / "data" / "sod.npz").exists() and (BASE_DIR / "animations").is_dir():
    ANIM_CFG["output_dir"] = str(BASE_DIR)
else:
    ANIM_CFG["output_dir"] = str(BASE_DIR / "stage1_animation_outputs")
print("Output:", ANIM_CFG["output_dir"])
print("Problems:", ANIM_CFG["problems"])

## 表示時刻と数値積分の時刻を分ける

各方式は元のCFL条件に従って独立に時間を進めます。**動画のフレーム時刻に合わせて$\Delta t$を切り詰めません。**
特に古典的LFでは散逸係数に$\Delta x/\Delta t$が入るため、出力の都合で時間刻みを変えると比較対象そのものが変わってしまいます。

共通フレーム時刻$t_f$を挟む二つの受理状態$\boldsymbol U^n,\boldsymbol U^{n+1}$から、

$$\boldsymbol U_{\rm frame}(t_f)=(1-\theta)\boldsymbol U^n+\theta\boldsymbol U^{n+1},\qquad
\theta=\frac{t_f-t_n}{t_{n+1}-t_n}$$

を**表示専用**に作ります。この補間状態を次の数値積分には使用しません。補間は保存変数に対して行い、その後で速度・圧力を求めます。
初期フレームと最終フレームは元の数値状態をそのまま使います。時間補間が波面をわずかに平滑化し得るので、定量的な誤差評価には元の保存状態を使ってください。

負圧などで停止した方式は、最後に受理された時刻より後を`NaN`にします。古い状態を現在時刻の解として表示することも、停止後に外挿することもありません。
動画の`STOPPED at`は**最後の受理時刻**、下の`rejected trial`は**棄却された試行の終端時刻**で、両者は区別します。

全方式で軸の範囲を固定し、色・線種・マーカーも問題間で統一します。密度・圧力の下限値補正や他ソルバーへの自動切替は行いません。


In [ ]:
"""Record common-time frames without changing a solver's integration steps."""
from pathlib import Path
import hashlib
import json
import numpy as np



def _signature(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=True).encode()).hexdigest()


def frame_cache_parameters(key, cfg):
    """Physical and sampling settings. Rendering-only changes do not rerun the simulation."""
    p = PROBLEMS[key]
    names = ("gamma", "cfl", "wave_speeds", "roe_entropy", "n_frames", "n_reference", "reference_limiter", "verify_observer")
    result = {name: cfg[name] for name in names}
    result.update({"format_version": 1, "problem": vars(p), "n": cfg["n_shu_osher"] if key == "shu_osher" else cfg["n_basic"]})
    result["solver_revision"] = "reference-and-order-metadata-v3"
    # Change cache_revision after editing the physical solver in the notebook.
    result["cache_revision"] = cfg.get("cache_revision", "original-solver-with-observer-v1")
    return json.loads(json.dumps(result))


class FrameRecorder:
    """Interpolate conserved variables for display only; no feedback to the integrator."""
    def __init__(self, times):
        self.times = np.asarray(times, dtype=float)
        if self.times.ndim != 1 or len(self.times) < 2 or self.times[0] != 0.0 or np.any(np.diff(self.times) <= 0.0):
            raise ValueError("Frame times must start at zero and increase strictly.")
        self.states = None
        self.left_time = np.full(len(self.times), np.nan)
        self.right_time = np.full(len(self.times), np.nan)
        self.weight = np.full(len(self.times), np.nan)
        self.previous_time = None
        self.previous_state = None
        self.next_frame = 0

    def __call__(self, t, U):
        if self.states is None:
            self.states = np.full((len(self.times),) + U.shape, np.nan)
            self.states[0] = U
            self.left_time[0] = self.right_time[0] = t
            self.weight[0] = 0.0
            self.next_frame = 1
        else:
            if t <= self.previous_time:
                raise ValueError("Accepted solver times must increase strictly.")
            while self.next_frame < len(self.times) and self.times[self.next_frame] <= t:
                j = self.next_frame
                target = self.times[j]
                w = (target - self.previous_time) / (t - self.previous_time)
                if not (0.0 <= w <= 1.0):
                    raise RuntimeError("A movie frame would require extrapolation.")
                # Preserve endpoint states exactly, including the final numerical solution.
                self.states[j] = U if target == t else (self.previous_state if target == self.previous_time else (1.0 - w) * self.previous_state + w * U)
                self.left_time[j], self.right_time[j], self.weight[j] = self.previous_time, t, w
                self.next_frame += 1
        self.previous_time = float(t)
        self.previous_state = U.copy()

    @property
    def valid(self):
        return np.isfinite(self.weight)


def run_summary(run):
    return {"solver": run.solver, "label": run.label, "order": run.space_order_nominal,
            "reconstruction_order": run.reconstruction_order, "space_order_nominal": run.space_order_nominal,
            "time_order": run.time_order, "n": run.n, "status": run.status,
            "t_reached": run.t, "t_target": run.target_time, "steps": run.steps, "seconds": run.elapsed,
            "fail_time": run.fail_time, "fail_dt": run.fail_dt, "fail_min_rho": run.fail_min_rho,
            "fail_min_p": run.fail_min_p, "fail_reason": run.fail_reason,
            "balance_max": float(np.max(np.abs(run.balance)))}


def _run_and_record(problem, solver, n, times, cfg, order=1, limiter="minmod"):
    recorder = FrameRecorder(times)
    kwargs = dict(n=n, gamma=cfg["gamma"], cfl=cfg["cfl"], order=order, limiter=limiter, wave_speeds=cfg["wave_speeds"], roe_entropy=cfg["roe_entropy"])
    run = simulate(problem, solver, observer=recorder, **kwargs)
    summary = run_summary(run)
    good = recorder.valid
    if not np.array_equal(recorder.states[0], run.U0):
        raise AssertionError("Initial movie frame differs from the solver initial state.")
    if np.any(good):
        cons_to_prim(recorder.states[good], cfg["gamma"])
        assert np.all(recorder.times[good] <= run.t)
    if run.status == "ok":
        assert good.all() and np.array_equal(recorder.states[-1], run.U)
    else:
        assert np.isnan(recorder.states[~good]).all()
    if cfg["verify_observer"] and order == 1:
        baseline = simulate(problem, solver, **kwargs)
        same = (np.array_equal(run.U, baseline.U) and run.steps == baseline.steps and run.status == baseline.status and run.t == baseline.t and np.array_equal(run.boundary_exchange, baseline.boundary_exchange))
        if not same:
            raise AssertionError("The observer changed the numerical evolution.")
        summary["observer_identical"] = True
    return run, recorder, summary


def save_animation_data(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {"times": data["times"], "x": data["x"], "edges": data["edges"], "reference": data["reference"]}
    for solver in ALL_SOLVERS:
        arrays["U_" + solver] = data["states"][solver]
        arrays["valid_" + solver] = data["valid"][solver]
        arrays["left_" + solver], arrays["right_" + solver], arrays["weight_" + solver] = data["brackets"][solver]
    metadata = {k: data[k] for k in ("key", "title", "gamma", "cfl", "n", "parameters", "data_id", "runs", "reference_label", "reference_check")}
    arrays["metadata"] = np.array(json.dumps(metadata, ensure_ascii=False))
    np.savez_compressed(path, **arrays)


def load_animation_data(path):
    with np.load(path, allow_pickle=False) as z:
        data = json.loads(str(z["metadata"]))
        data.update({name: z[name].copy() for name in ("times", "x", "edges", "reference")})
        data["states"] = {s: z["U_" + s].copy() for s in ALL_SOLVERS}
        data["valid"] = {s: z["valid_" + s].copy() for s in ALL_SOLVERS}
        data["brackets"] = {s: tuple(z[p + s].copy() for p in ("left_", "right_", "weight_")) for s in ALL_SOLVERS}
    return data


def get_animation_data(key, cfg=ANIM_CFG):
    """Six Stage-1 schemes, common output times, and an exact or numerical reference."""
    parameters = frame_cache_parameters(key, cfg)
    cache = Path(cfg["output_dir"]) / "data" / (key + ".npz")
    if cfg["reuse_data"] and cache.exists():
        cached = load_animation_data(cache)
        if cached["parameters"] == parameters:
            print(f"{key}: read cached common-time frames")
            return cached
    p, n = PROBLEMS[key], parameters["n"]
    times = np.linspace(0.0, p.final_time, int(cfg["n_frames"]))
    data = {"key": key, "title": p.title, "gamma": cfg["gamma"], "cfl": cfg["cfl"], "n": n,
            "times": times, "parameters": parameters, "data_id": _signature(parameters),
            "runs": {}, "states": {}, "valid": {}, "brackets": {}, "reference_check": None}
    for solver in ALL_SOLVERS:
        run, recorder, summary = _run_and_record(p, solver, n, times, cfg)
        data["runs"][solver], data["states"][solver], data["valid"][solver] = summary, recorder.states, recorder.valid
        data["brackets"][solver] = (recorder.left_time, recorder.right_time, recorder.weight)
        data["x"], data["edges"] = run.x, run.edges
    if key != "shu_osher":
        data["reference"] = np.stack([exact_cell_average(p, data["edges"], t, cfg["gamma"]) for t in times])
        data["reference_label"] = "Exact (cell-averaged U)"
    else:
        nr1, nr2 = map(int, cfg["n_reference"])
        if not (nr2 > nr1 >= n and nr2 % nr1 == 0 and nr1 % n == 0):
            raise ValueError("Require n_reference[1] > n_reference[0] >= n_shu_osher and integer refinement ratios.")
        averaged = []
        for nr in (nr1, nr2):
            print(f"{key}: numerical reference, HLLC + MUSCL/SSP-RK2, N={nr}")
            ref_path = cache.parent / f"shu_osher_reference_{nr}_{data['data_id'][:12]}.npz"
            if cfg["reuse_data"] and ref_path.exists():
                with np.load(ref_path, allow_pickle=False) as saved:
                    projected = saved["projected"].copy()
            else:
                run, recorder, summary = _run_and_record(p, "HLLC", nr, times, cfg, order=2, limiter=cfg["reference_limiter"])
                if run.status != "ok":
                    raise RuntimeError("A numerical reference failed: " + run.fail_reason)
                projected = recorder.states.reshape(len(times), n, nr // n, 3).mean(axis=2)
                ref_path.parent.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(ref_path, projected=projected)
            averaged.append(projected)
        data["reference"] = averaged[-1]
        gap = np.mean(np.abs(averaged[1][..., 0] - averaged[0][..., 0]), axis=1)
        data["reference_label"] = f"Numerical ref. N={nr2}\n(projected to N={n})"
        data["reference_check"] = {"n_reference": [nr1, nr2], "final_density_L1_gap": float(gap[-1]), "max_movie_density_L1_gap": float(gap.max()), "note": "Two-resolution numerical comparison, not an exact solution or a convergence proof."}
    cons_to_prim(data["reference"], cfg["gamma"])
    save_animation_data(data, cache)
    print(f"{key}: " + ", ".join(s + "=" + data["runs"][s]["status"] for s in ALL_SOLVERS))
    return data

## アニメーション・MP4・HTML一覧の作成関数

一つの動画では一つの物理量を一つの座標軸に表示します。MP4保存では固定した背景を再利用し、曲線・時刻・状態表示だけを描き直します。

In [ ]:
"""Single-panel comparison animations, MP4 export, and a self-contained HTML gallery."""
from pathlib import Path
import base64
import hashlib
import html
import json
import shutil
import subprocess
import tempfile
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, Video, display

# 岡部・伊藤パレット。元のNotebookと方式ごとの対応を変えない。
ANIM_COLORS = {"LF": "#0072B2", "LW": "#D55E00", "Roe": "#009E73", "LLF": "#CC79A7", "HLL": "#E69F00", "HLLC": "#56B4E9"}
ANIM_LINES = {"LF": "-", "LW": "-", "Roe": "--", "LLF": "-.", "HLL": ":", "HLLC": "--"}
ANIM_MARKERS = {"LF": "s", "LW": "^", "Roe": "o", "LLF": "D", "HLL": "v", "HLLC": "x"}
FIELD_LABELS = {"rho": r"Density $\rho$", "u": r"Velocity $u$", "p": r"Pressure $p$", "e": r"Specific internal energy $e$"}
FIELD_JA = {"rho": "密度", "u": "速度", "p": "圧力", "e": "比内部エネルギー"}
CASE_JA = {"entropy": "滑らかなエントロピー波", "contact_static": "静止する接触不連続面", "contact": "移動する接触不連続面", "sod": "Sod衝撃波管", "double_rarefaction": "二重希薄波", "shu_osher": "Shu–Osher問題", "lax": "Lax衝撃波管（追加）"}
CASE_NOTES = {
    "entropy": "密度の正弦波が1周します。振幅の低下と位相のずれを比較してください。",
    "contact_static": "密度だけが不連続な静止状態です。接触面が数値拡散で広がるかを確認します。",
    "contact": "密度の矩形波が右へ移動します。境界の広がりとLWの数値振動を比較します。",
    "sod": "左向きの希薄波・接触不連続面・右向きの衝撃波の発生と伝播を比べます。",
    "double_rarefaction": "中央で密度・圧力が低下します。負圧などで停止した方式は、最後の受理時刻を超えて描画しません。密度・圧力は対数軸です。",
    "shu_osher": "衝撃波後方の短波長の密度変動を比較します。黒線は高解像度の数値参照解であり、厳密解ではありません。",
    "lax": "Sodとは異なる速度・圧力差を持つ追加の衝撃波管問題です。",
}


def animation_field(U, field, gamma):
    W = cons_to_prim(U, gamma, check=False)
    if field in ("rho", "u", "p"):
        return W[..., ("rho", "u", "p").index(field)]
    if field == "e":
        return W[..., 2] / ((gamma - 1.0) * W[..., 0])
    raise ValueError("field must be rho, u, p, or e.")


def make_comparison_animation(data, field="rho", cfg=ANIM_CFG, solvers=None, xlim=None, ylim=None, yscale=None):
    """各動画は一つの座標軸。全方式は同じ物理時刻・同じ固定軸範囲で表示する。"""
    solvers = ALL_SOLVERS if solvers is None else tuple(solvers)
    if not solvers or any(s not in ALL_SOLVERS for s in solvers):
        raise ValueError("Select at least one known solver.")
    if field not in FIELD_LABELS or cfg["fps"] <= 0:
        raise ValueError("Invalid field or frame rate.")
    for name in ("hold_start", "hold_end"):
        if cfg[name] < 0 or int(cfg[name]) != cfg[name]:
            raise ValueError("Hold counts must be nonnegative integers.")
    times, x = data["times"], data["x"]
    values = {s: animation_field(data["states"][s], field, data["gamma"]) for s in solvers}
    reference = animation_field(data["reference"], field, data["gamma"])
    bounds = (float(data["edges"][0]), float(data["edges"][-1])) if xlim is None else tuple(xlim)
    visible_x = (x >= bounds[0]) & (x <= bounds[1])
    if not visible_x.any():
        raise ValueError("xlim does not contain any cells.")
    scale = ("log" if data["key"] == "double_rarefaction" and field in ("rho", "p") else "linear") if yscale is None else yscale
    extrema = [reference[:, visible_x].ravel()] + [values[s][data["valid"][s]][:, visible_x].ravel() for s in solvers]
    low = min(float(np.nanmin(a)) for a in extrema)
    high = max(float(np.nanmax(a)) for a in extrema)
    if ylim is None:
        if scale == "log":
            if low <= 0.0:
                raise ValueError("Logarithmic movies require positive accepted values.")
            yrange = (low / 1.45, high * 1.45)
        else:
            span = high - low
            margin = 0.09 * span if span > 1e-10 * max(1.0, abs(low), abs(high)) else 0.05 * max(1.0, abs(low), abs(high))
            yrange = (low - margin, high + margin)
    else:
        yrange = tuple(ylim)
    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 11, "axes.labelsize": 12, "axes.titlesize": 12, "figure.dpi": cfg["dpi"]}):
        fig = plt.figure(figsize=cfg["figsize"], dpi=cfg["dpi"])
        ax = fig.add_axes((0.085, 0.22, 0.66, 0.60))
        fig.text(0.085, 0.945, data["title"], fontsize=17, weight="bold")
        clock = fig.text(0.085, 0.895, "", fontsize=12)
        ref_line, = ax.plot(x, reference[0], color="#000000", lw=1.65, label=data["reference_label"], zorder=10)
        lines = {}
        for i, s in enumerate(solvers):
            stride = max(1, data["n"] // 19)
            lines[s], = ax.plot(x, values[s][0], color=ANIM_COLORS[s], ls=ANIM_LINES[s], lw=1.65, marker=ANIM_MARKERS[s], ms=4.0, markevery=(i % stride, stride), mfc="none", label=("LLF (Rusanov)" if s == "LLF" else data["runs"][s]["label"]))
        ax.set(xlabel="$x$", ylabel=FIELD_LABELS[field], xlim=bounds, ylim=yrange, yscale=scale)
        ax.grid(alpha=0.22, which="both")
        ax.tick_params(direction="out")
        ax.legend(loc="upper left", bbox_to_anchor=(1.025, 1.025), borderaxespad=0.0, frameon=False, fontsize=10, handlelength=2.5, labelspacing=0.65)
        fig.text(0.775, 0.45, "Solver status", fontsize=11, weight="bold")
        status_lines = {s: fig.text(0.775, 0.413 - 0.034 * i, "", fontsize=9.5, color=ANIM_COLORS[s]) for i, s in enumerate(solvers)}
        failure_note = fig.text(0.085, 0.115 if data["key"] == "double_rarefaction" else 0.135, "", fontsize=10)
        fig.text(0.085, 0.083, "LF: classical; Roe/LLF/HLL/HLLC: 1st order; LW: Richtmyer two-stage, 2nd order.", fontsize=9.5)
        fig.text(0.085, 0.050, "Common display times: conserved states are interpolated only for plotting; solver time steps are unchanged.", fontsize=9)
        fig.text(0.085, 0.020, "Fixed axes throughout. Markers denote cells; styles are fixed across problems. Failed runs are never extrapolated.", fontsize=9)

        def update(j):
            t = float(times[j])
            clock.set_text(f"Stage 1  |  N = {data['n']}  |  CFL = {data['cfl']:g}  |  t = {t:.6f} / {times[-1]:g}  |  frame {j + 1}/{len(times)}")
            ref_line.set_ydata(reference[j])
            notes = []
            for s in solvers:
                info = data["runs"][s]
                display_name = "LLF (Rusanov)" if s == "LLF" else s
                if data["valid"][s][j]:
                    lines[s].set_ydata(values[s][j])
                    status_lines[s].set_text(f"{display_name}: {'complete' if j == len(times) - 1 else 'active'}")
                else:
                    lines[s].set_ydata(np.full_like(x, np.nan))
                    status_lines[s].set_text(f"{display_name}: STOPPED at {info['t_reached']:.5g}")
                    trial = info.get("fail_time", float("nan"))
                    pmin = info.get("fail_min_p", float("nan"))
                    reason = f"{display_name}: last accepted t={info['t_reached']:.5g}; rejected trial t={trial:.5g}"
                    if np.isfinite(pmin):
                        reason += f", min(p)={pmin:.3g}"
                    notes.append(reason)
            failure_note.set_text("\n".join(notes) if notes else "All displayed states are admissible. No density/pressure floor or automatic solver fallback.")
            return [ref_line, clock, failure_note] + list(lines.values()) + list(status_lines.values())

        frames = [0] * int(cfg["hold_start"]) + list(range(len(times))) + [len(times) - 1] * int(cfg["hold_end"])
        movie = animation.FuncAnimation(fig, update, frames=frames, interval=1000.0 / cfg["fps"], repeat=True, blit=False)
        update(0)
    return fig, movie, update


def configure_ffmpeg():
    """Use system ffmpeg, or imageio-ffmpeg when installed. No shell-specific commands."""
    if animation.writers.is_available("ffmpeg"):
        return True
    path = shutil.which("ffmpeg")
    if path:
        matplotlib.rcParams["animation.ffmpeg_path"] = path
    else:
        try:
            import imageio_ffmpeg
            matplotlib.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
        except (ImportError, RuntimeError):
            return False
    return animation.writers.is_available("ffmpeg")


def export_animation(data, field="rho", cfg=ANIM_CFG, solvers=None, xlim=None, ylim=None, yscale=None, name_suffix=""):
    """Create/reuse an MP4. Return None when MP4 export is disabled or ffmpeg is absent."""
    if not cfg["save_mp4"] or not configure_ffmpeg():
        return None
    folder = Path(cfg["output_dir"]) / "animations"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"stage1_{data['key']}_{field}{name_suffix}.mp4"
    settings = {k: cfg[k] for k in ("fps", "dpi", "figsize", "hold_start", "hold_end")}
    settings.update({"data_id": data["data_id"], "field": field, "solvers": solvers, "xlim": xlim, "ylim": ylim, "yscale": yscale, "render_version": 4})
    digest = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
    stamp = path.with_suffix(".json")
    if cfg["reuse_movies"] and path.exists() and stamp.exists():
        if json.loads(stamp.read_text())["render_id"] == digest:
            return path
    fig, movie, update = make_comparison_animation(data, field, cfg, solvers=solvers, xlim=xlim, ylim=ylim, yscale=yscale)
    # Draw ticks, grid, labels and legend once; redraw only the curves and changing text.
    # Streaming raw RGB(A) frames directly avoids a complete savefig() for each movie frame.
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    FigureCanvasAgg(fig)
    moving = update(0)
    for artist in moving:
        artist.set_animated(True)
    fig.canvas.draw()
    background = fig.canvas.copy_from_bbox(fig.bbox)
    width, height = fig.canvas.get_width_height()
    temp_path = path.with_name(path.stem + ".tmp.mp4")
    command = [str(matplotlib.rcParams["animation.ffmpeg_path"]), "-y", "-loglevel", "error", "-f", "rawvideo", "-vcodec", "rawvideo", "-s", f"{width}x{height}", "-pix_fmt", "rgba", "-framerate", str(cfg["fps"]), "-i", "pipe:0", "-an", "-c:v", "libx264", "-crf", "20", "-preset", "fast", "-threads", "2", "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-metadata", "title=" + data["title"], str(temp_path)]
    sequence = [0] * int(cfg["hold_start"]) + list(range(len(data["times"]))) + [len(data["times"]) - 1] * int(cfg["hold_end"])
    with tempfile.TemporaryFile() as error_log:
        process = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=error_log)
        try:
            for frame in sequence:
                fig.canvas.restore_region(background)
                for artist in sorted(update(frame), key=lambda a: a.get_zorder()):
                    fig.draw_artist(artist)
                process.stdin.write(fig.canvas.buffer_rgba())
            process.stdin.close()
            code = process.wait()
            if code:
                error_log.seek(0)
                raise RuntimeError("ffmpeg failed: " + error_log.read().decode(errors="replace"))
            temp_path.replace(path)
        finally:
            if process.poll() is None:
                process.kill()
                process.wait()
            if process.stdin and not process.stdin.closed:
                process.stdin.close()
            if temp_path.exists():
                temp_path.unlink()
            plt.close(fig)
    stamp.write_text(json.dumps({"render_id": digest, "settings": settings, "frames": len(data["times"]) + cfg["hold_start"] + cfg["hold_end"], "seconds": (len(data["times"]) + cfg["hold_start"] + cfg["hold_end"]) / cfg["fps"]}, indent=2))
    print(f"Saved: {path.name}")
    return path


def show_animation(data, field="rho", cfg=ANIM_CFG, solvers=None, xlim=None, ylim=None, yscale=None, name_suffix=""):
    """MP4 playback by default; JavaScript controls are available without ffmpeg."""
    path = export_animation(data, field, cfg, solvers=solvers, xlim=xlim, ylim=ylim, yscale=yscale, name_suffix=name_suffix)
    if path is not None:
        display(Video(filename=str(path), embed=True, width=1050, html_attributes="controls loop playsinline preload='metadata'"))
        return path
    fig, movie, _ = make_comparison_animation(data, field, cfg, solvers=solvers, xlim=xlim, ylim=ylim, yscale=yscale)
    # A JSHTML frame is larger than a compressed video; limit the resolution only in this fallback.
    fig.set_dpi(min(80, cfg["dpi"]))
    try:
        with matplotlib.rc_context({"animation.embed_limit": 160}):
            markup = movie.to_jshtml(fps=cfg["fps"], default_mode="loop")
        display(HTML(markup))
    finally:
        plt.close(fig)
    return None


def export_gallery(datasets, cfg=ANIM_CFG, embed=True):
    """One video at a time; switch problem/quantity without Python or an Internet connection."""
    folder = Path(cfg["output_dir"])
    items = []
    for key, data in datasets.items():
        fields = []
        for field in cfg["fields_by_problem"][key]:
            path = folder / "animations" / f"stage1_{key}_{field}.mp4"
            if not path.exists():
                continue
            source = "data:video/mp4;base64," + base64.b64encode(path.read_bytes()).decode() if embed else "animations/" + path.name
            fields.append({"key": field, "name": FIELD_JA[field], "src": source, "file": path.name})
        if fields:
            stopped = []
            for s in ALL_SOLVERS:
                r = data["runs"][s]
                if r["status"] != "ok":
                    stopped.append(f"{s}：最後の受理時刻 t={r['t_reached']:.8g}、棄却した試行の終端 t={r['fail_time']:.8g}、min(p)={r['fail_min_p']:.6g}。")
            items.append({"key": key, "title": CASE_JA[key], "note": CASE_NOTES[key], "n": data["n"], "end": float(data["times"][-1]), "fields": fields, "stopped": stopped})
    if not items:
        print("No MP4 files exist yet; the gallery was not written.")
        return None
    payload = json.dumps(items, ensure_ascii=False).replace("</", "<\\/")
    page = '''<!doctype html><html lang="ja"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Stage 1 | 1D Euler scheme animations</title>
<style>body{font-family:system-ui,-apple-system,sans-serif;max-width:1180px;margin:28px auto;padding:0 20px;color:#182432;background:#f5f7fa}h1{font-size:27px;margin-bottom:8px}p{line-height:1.7}.box{background:#fff;border:1px solid #d9e0e8;border-radius:12px;padding:20px;margin-top:18px}label{display:inline-flex;gap:10px;align-items:center;margin:5px 18px 5px 0}select,button{padding:8px;font-size:15px;border:1px solid #acb9c7;border-radius:5px;background:white}video{width:100%;height:auto;background:white;margin-top:10px}small{line-height:1.7;color:#465666}#failure{background:#fff4ec;padding:12px;border-left:4px solid #b75b19;white-space:pre-line}a{color:#006ba2}.badge{font-size:13px;letter-spacing:1px;color:#38637f}.key{font-family:monospace}</style>
<div class="badge">COMPUTATIONAL FLUID DYNAMICS · STAGE 1</div><h1>1次元Euler方程式：スキームの時間発展比較</h1>
<p>LF・LW・Roe・LLF (Rusanov)・HLL（HLLE）・HLLCを、同じ物理時刻と同じ軸範囲で比較します。保存済み動画なので再生にPythonやインターネット接続は不要です。</p>
<div class="box"><label>問題<select id="case"></select></label><label>表示量<select id="field"></select></label><label>再生速度<select id="speed"><option value="0.25">0.25×</option><option value="0.5">0.5×</option><option value="1" selected>1×</option><option value="2">2×</option></select></label><label><input id="loop" type="checkbox" checked>繰り返す</label>
<h2 id="title"></h2><p id="note"></p><video id="video" controls loop playsinline preload="metadata"></video><p><span id="meta"></span>　<a id="download" download>このMP4を保存</a></p><p id="failure" hidden></p></div>
<div class="box"><h2>比較時の注意</h2><p>黒線は厳密リーマン解などの保存変数を同じ格子でセル平均した参照値です。Shu–OsherのみHLLC＋MUSCL/SSP-RK2の高解像度数値参照解です。LF・Roe・LLF・HLL・HLLCは1次精度の構成、LWはRichtmyerの2段階・2次精度法です。</p><p>動画の出力時刻は共通ですが、各方式のCFL時間刻みは変更していません。フレーム時刻を挟む二つの受理済み保存状態を表示用に線形補間しています。補間状態で計算を再開することはありません。補間は衝撃波面をわずかに平滑化し得るため、定量評価には元の数値解を用いてください。停止後の状態を外挿・再利用して現在時刻の曲線として描くことはありません。</p><small>動画の最初と最後には静止時間があります。物理時刻は図の上部に表示されます。各問題の物理的な終了時刻は異なります。動画の長さを揃えているので、異なる問題間の再生速度は同一の物理時間尺度ではありません。</small></div>
<script>const items=__DATA__;const c=document.getElementById('case'),f=document.getElementById('field'),v=document.getElementById('video'),s=document.getElementById('speed');for(const a of items)c.add(new Option(a.title,a.key));
function selectField(){const a=items.find(x=>x.key===c.value),b=a.fields.find(x=>x.key===f.value);v.pause();v.src=b.src;v.load();v.playbackRate=Number(s.value);document.getElementById('meta').textContent='N = '+a.n+' | t_end = '+a.end+' | '+b.name;const d=document.getElementById('download');d.href=b.src;d.download=b.file;}
function selectCase(){const a=items.find(x=>x.key===c.value);f.replaceChildren();for(const b of a.fields)f.add(new Option(b.name,b.key));document.getElementById('title').textContent=a.title;document.getElementById('note').textContent=a.note;const z=document.getElementById('failure');z.hidden=!a.stopped.length;z.textContent=a.stopped.join(String.fromCharCode(10));selectField();}
c.onchange=selectCase;f.onchange=selectField;s.onchange=()=>{v.playbackRate=Number(s.value)};document.getElementById('loop').onchange=e=>{v.loop=e.target.checked};selectCase();</script></html>'''
    page = page.replace("__DATA__", payload)
    path = folder / ("stage1_gallery.html" if embed else "index.html")
    path.write_text(page, encoding="utf-8")
    print(f"Saved gallery: {path.name}")
    return path

## 共通時刻のスナップショットを計算・保存する

`verify_observer=True`では各方式を記録なしでも計算し、最終状態・ステップ数・境界流束が変わっていないことを検査します。Shu–Osherの数値参照解は1,600・3,200セルで計算し、比較格子へ保存変数を体積平均した密度の差も記録します。

In [ ]:
ANIMATION_DATA = {key: get_animation_data(key, ANIM_CFG) for key in ANIM_CFG["problems"]}
for key, data in ANIMATION_DATA.items():
    print(key, {solver: info["status"] for solver, info in data["runs"].items()})
    if data["reference_check"] is not None:
        print("Numerical-reference check:", data["reference_check"])

## 検証結果

`observer_identical=True`は、表示用の記録を付けても数値積分の結果が変わらなかったことを意味します。低密度問題での停止は比較結果の一部であり、Notebookの実行エラーではありません。

In [ ]:
rows = [dict(problem=key, **info) for key, data in ANIMATION_DATA.items() for info in data["runs"].values()]
print("Cases:", len(ANIMATION_DATA), "Runs:", len(rows))
print("Completed:", sum(row["status"] == "ok" for row in rows), "Stopped:", sum(row["status"] != "ok" for row in rows))
print("Observer checks:", sum(row.get("observer_identical", False) for row in rows))
for row in rows:
    if row["status"] != "ok":
        print(row["problem"], row["solver"], "last accepted =", row["t_reached"], "rejected trial =", row["fail_time"], "min(p) =", row["fail_min_p"])

## 動画1：滑らかなエントロピー波

密度分布が1周する過程で、振幅の減衰と位相のずれを比べます。

In [ ]:
if "entropy" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["entropy"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["entropy"], field, ANIM_CFG)

## 動画2：静止する接触不連続面

不連続面が移動しない条件です。Roe・HLLCだけでなく、この特別な状態ではLWも接触面を保持します。

In [ ]:
if "contact_static" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["contact_static"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["contact_static"], field, ANIM_CFG)

## 動画3：移動する接触不連続面

矩形波の両端を追い、数値拡散とLWのovershoot/undershootを区別します。

In [ ]:
if "contact" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["contact"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["contact"], field, ANIM_CFG)

## 動画4：Sod衝撃波管

希薄波・接触面・衝撃波が形成されて移動する過程を、密度・速度・圧力で比較します。

In [ ]:
if "sod" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["sod"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["sod"], field, ANIM_CFG)

## 動画5：二重希薄波

密度・圧力の動画では固定した対数軸を使います。速度・比内部エネルギーは線形軸です。標準条件ではLWとRoeが初期に負圧を生じて停止するので、その後は4方式だけの曲線が残ります。

In [ ]:
if "double_rarefaction" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["double_rarefaction"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["double_rarefaction"], field, ANIM_CFG)

## 動画6：Shu–Osher問題

黒線は3,200セルのHLLC＋MUSCL/SSP-RK2（MCリミッター）の数値解を400セルへ投影したものです。1,600セルとの比較は参照解の不確かさの目安であり、厳密解や収束の証明ではありません。

In [ ]:
if "shu_osher" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["shu_osher"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["shu_osher"], field, ANIM_CFG)

## 動画7：Lax衝撃波管（追加）

元のNotebookの追加実験も、同じStage 1の6方式で動画化しました。

In [ ]:
if "lax" in ANIMATION_DATA:
    for field in ANIM_CFG["fields_by_problem"]["lax"]:
        display(HTML("<h4>" + FIELD_JA[field] + "</h4>"))
        show_animation(ANIMATION_DATA["lax"], field, ANIM_CFG)

## 問題・物理量を切り替える動画一覧

MP4が保存されていれば、全動画を埋め込んだ`stage1_gallery.html`を作ります。このHTMLをブラウザーで開くと、問題・物理量・再生速度を切り替えられます。
小さい`index.html`はMP4ファイルへの相対パスを使うため、`animations`フォルダーと一緒に移動してください。

In [ ]:
embedded_gallery = export_gallery(ANIMATION_DATA, ANIM_CFG, embed=True)
linked_gallery = export_gallery(ANIMATION_DATA, ANIM_CFG, embed=False)
print("Gallery:", embedded_gallery)

## 表示を変更する例（必要なセルだけ実行）

重なった曲線を区別するには方式を絞り、接触面の近くを見るには`xlim`を指定します。元のMP4を上書きしないように`name_suffix`も指定します。

```python
show_animation(ANIMATION_DATA["sod"], "rho", ANIM_CFG, solvers=("Roe", "HLL", "HLLC"), xlim=(0.4, 0.95), name_suffix="_contact_zoom")
show_animation(ANIMATION_DATA["double_rarefaction"], "p", ANIM_CFG, yscale="linear", name_suffix="_linear")
```

ffmpegがない環境でMP4を新しく書き出す場合、Notebookのセルで`%pip install imageio-ffmpeg`を実行すると、利用可能な環境では同パッケージが提供するffmpegを自動検出します。
これは再生成時の補助であり、保存済みMP4の再生には必要ありません。JavaScriptによるコマ送りを使う場合は`ANIM_CFG["save_mp4"]=False`にします。

**NotebookのHTMLや動画出力が表示されない場合**は、JupyterでNotebookを信頼済みにして開くか、該当セルを再実行してください。独立したHTMLまたはMP4でも確認できます。

### 実装の確認先

元のNotebookの流体スキームの参考文献に加え、動画表示には以下の公式資料を参照しています。

- Matplotlib, `FuncAnimation`: https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html
- Matplotlib, animation API: https://matplotlib.org/stable/api/animation_api.html
- Matplotlib, `FFMpegWriter`: https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FFMpegWriter.html
